# Fillterizetion: filtered `z_tracking.csv` kinematics

Standalone filtered kinematics notebook.

**Safety contract**
- Reads the raw experiment tree and existing Python analysis module only.
- Uses `z_tracking.csv` files for side-Z input; it does **not** decode/open video files.
- Writes all new CSVs and figures under `analysis/Kinematics/result_fillter/`.
- Does not edit `kinematic_analysis.ipynb`, `kinematics_analysis.py`, `analysis/Kinematics/results/`, or raw experiment folders.

The spelling `fillter` / `result_fillter` is kept intentionally to match the requested folder/tag name.


In [4]:
from pathlib import Path
import importlib
import sys
import pandas as pd
import numpy as np
from IPython.display import display


def find_analysis_dir(start: Path | None = None) -> Path:
    """Locate analysis/Kinematics from the notebook's current working directory.

    Jupyter/VS Code may start the kernel from the project root, the notebook
    folder, or an intermediate folder such as ``analysis``.  Walk upward so
    the setup cell works from any directory inside the repository.
    """
    current = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "kinematics_analysis.py").exists():
            return candidate
        nested = candidate / "analysis" / "Kinematics"
        if (nested / "kinematics_analysis.py").exists():
            return nested
    raise FileNotFoundError(
        "Could not find analysis/Kinematics/kinematics_analysis.py. "
        "Start the notebook from somewhere inside the Parallel_Heptics repository."
    )


ANALYSIS_DIR = find_analysis_dir()
analysis_dir_text = str(ANALYSIS_DIR)
if analysis_dir_text not in sys.path:
    sys.path.insert(0, analysis_dir_text)
import kinematics_analysis as ka
# Pick up edits to kinematics_analysis.py without restarting the kernel. Jupyter
# caches the first import, so without this a re-run silently uses stale code.
importlib.reload(ka)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# --- Raw data + selection -------------------------------------------------
# Read-only raw data root: the current Parallel_Heptics results tree (one folder
# per subject, each holding answers.csv + pair_NNN/tracking.csv + side/top camera
# mp4). This is the same root the psychophysics notebook reads.
DATA_ROOT = Path(r"C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\results")

# Choose what to analyze, mirroring the psychophysics notebook:
#   single subject  -> "N_E_1"   (or a one-element list ["N_E_1"] -> same folder)
#   a whole group   -> "N_E" / "L_E" / "N_P" / "L_P"
#   combined groups -> "L_N_E"   (= L_E + N_E)
#   only filtered folders -> "FILTER_ONLY" (or "L_E_FILTER_ONLY" for one group)
#   a hand-picked list -> ["L_E_1", "L_E_2", ...] (each subject + one custom aggregate folder)
#   every valid subject -> None
# Folders carrying a "not"/"old"/"unfinished" token are skipped automatically.
DATA_SELECTION = "L_N_E"
# True: group/cohort selections skip folders marked "filter"; False: keep every valid called subject.
# Keep this False when DATA_SELECTION is FILTER_ONLY / <GROUP>_FILTER_ONLY.
# Explicit single-subject selections are kept in both cases.
EXCLUDE_FILTER_FOLDERS = True
SELECTION_LABEL = ka.selection_output_label(DATA_SELECTION, exclude_filter_folders=EXCLUDE_FILTER_FOLDERS)

# Output tree: the subject is the primary unit. Subjects and group folders are
# top-level siblings under results/, each with csv/ and figures/ split into the
# same semantic categories (filled in by ka.organize_kinematic_results_tree at
# the end of the run):
#   results/<subject>/{csv,figures}/<category>/   <- per-subject outputs
#   results/L_E , results/N_E , results/L_N_E /{csv,figures}/<category>/  <- named groups
#   results/<custom-selection>/{csv,figures}/<category>/  <- hand-picked aggregate
# A single-subject run (string OR one-element list) produces only that one folder.
#
# results/ is NOT wiped each run. Only the folders THIS run (re)makes are cleared
# of stale content: the selected subject/group folder here, plus each selected
# subject/group folder in the discovery cell. Folders are created lazily as
# outputs are written, so an interrupted run never leaves empty shells, and
# folders from other runs are left in place. Legacy results_old is untouched.
# There is no _group_level staging folder: a single-subject selection stays in
# results/<subject>/, and group selections stay in results/<group>/ or
# results/<custom-selection>/.
FILTER_TAG = "fillter"
RESULTS_ROOT = (ANALYSIS_DIR / "result_fillter").resolve()
RUN_OUTPUT_ROOT = (RESULTS_ROOT / SELECTION_LABEL).resolve()
OUTPUT_ROOT = RUN_OUTPUT_ROOT
FILTERED_Z_TRACKING_ROOT = (RESULTS_ROOT / "filtered_z_tracking_sources" / f"{SELECTION_LABEL}_{FILTER_TAG}").resolve()

# Keep False to avoid deleting previous filtered outputs by accident. If you want
# a clean filtered rerun, set True; it clears only paths inside result_fillter.
CLEAR_PREVIOUS_FILTER_OUTPUT = False

assert RESULTS_ROOT.name == "result_fillter", RESULTS_ROOT
assert OUTPUT_ROOT.is_relative_to(RESULTS_ROOT), (OUTPUT_ROOT, RESULTS_ROOT)
assert FILTERED_Z_TRACKING_ROOT.is_relative_to(RESULTS_ROOT), (FILTERED_Z_TRACKING_ROOT, RESULTS_ROOT)
if CLEAR_PREVIOUS_FILTER_OUTPUT:
    ka.reset_output_root(RUN_OUTPUT_ROOT)
    ka.reset_output_root(FILTERED_Z_TRACKING_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FILTERED_Z_TRACKING_ROOT.mkdir(parents=True, exist_ok=True)

CENTER_X = 320.0
CENTER_Y = 240.0
LOWPASS_CUTOFF_HZ = 3.0
HIGHPASS_CUTOFF_HZ = 0.10
TRAJECTORY_TIME_BINS = 50
SIDE_VIDEO_SAMPLES_PER_TRIAL = 30  # samples from filtered z_tracking.csv; no video decoding
MAX_TRACKING_TRIALS = None        # set a number for debugging only
MAX_SIDE_VIDEO_TRIALS = None      # set a number for debugging only
FIG_DPI = 160

# Filterization: per z_tracking trajectory, replace the upper 5% values with
# interpolated values and then apply a short centered rolling median (LPF-like).
UPPER_TAIL_FRACTION = 0.05
ROLLING_WINDOW_SAMPLES = 5
FILTER_VALUE_COLUMNS = tuple(getattr(ka, "Z_TRACKING_VALUE_COLUMNS", (
    "z_active_finger_lift_px",
    "z_hand_midpoint_lift_px",
    "z_thumb_lift_px",
)))

print("ANALYSIS_DIR", ANALYSIS_DIR.resolve())
print("DATA_ROOT", DATA_ROOT)
print("DATA_SELECTION", DATA_SELECTION, "->", SELECTION_LABEL)
print("EXCLUDE_FILTER_FOLDERS", EXCLUDE_FILTER_FOLDERS)
print("RUN_OUTPUT_ROOT", RUN_OUTPUT_ROOT.resolve())
print("OUTPUT_ROOT", OUTPUT_ROOT.resolve())
print("FILTERED_Z_TRACKING_ROOT", FILTERED_Z_TRACKING_ROOT.resolve())
print("FILTER_TAG", FILTER_TAG)


ANALYSIS_DIR C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics
DATA_ROOT C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\results
DATA_SELECTION L_N_E -> L_N_E
EXCLUDE_FILTER_FOLDERS True
RUN_OUTPUT_ROOT C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\result_fillter\L_N_E
OUTPUT_ROOT C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\result_fillter\L_N_E
FILTERED_Z_TRACKING_ROOT C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\result_fillter\filtered_z_tracking_sources\L_N_E_fillter
FILTER_TAG fillter


## 1b. Create filtered `z_tracking.csv` copies

This notebook writes filtered copies named `z_tracking_fillter.csv` into `result_fillter/filtered_z_tracking_sources/...`.
The raw `z_tracking.csv` files are never changed.

For each z value column, the filter:
1. Finds the 95th percentile inside that file/trajectory.
2. Marks values above that threshold as the maximum 5% spike set.
3. Replaces those spike values by interpolation.
4. Applies a short centered rolling median as a low-pass-like smoother.

The downstream side-Z loader is patched only in notebook memory so `estimate_side_video_z()` consumes these filtered CSV copies and never decodes video.


In [5]:
def _safe_relative_path(path: Path, root: Path) -> Path:
    path = Path(path).resolve()
    root = Path(root).resolve()
    try:
        return path.relative_to(root)
    except ValueError:
        return Path(ka.sanitize_name(str(path).replace(":", "")))


def fillter_z_tracking_frame(raw: pd.DataFrame) -> tuple[pd.DataFrame, list[dict[str, object]]]:
    """Filter upper-tail z spikes in one z_tracking trajectory.

    Original values are kept in `*_raw_unfilltered`; filtered values replace the
    original z columns so downstream kinematic calculations use the filtered
    trajectory.
    """
    out = raw.copy()
    out["fillter_tag"] = FILTER_TAG
    out["fillter_upper_tail_fraction"] = float(UPPER_TAIL_FRACTION)
    out["fillter_rolling_window_samples"] = int(ROLLING_WINDOW_SAMPLES or 1)
    stats: list[dict[str, object]] = []
    for col in FILTER_VALUE_COLUMNS:
        if col not in out.columns:
            continue
        values = pd.to_numeric(out[col], errors="coerce")
        valid = values.dropna()
        out[f"{col}_raw_unfilltered"] = values
        if valid.empty:
            out[f"{col}_fillter_removed_upper_tail"] = 0
            out[f"{col}_fillter_threshold_p95"] = np.nan
            stats.append({"column": col, "n_rows": len(out), "n_valid": 0, "threshold": np.nan, "n_removed": 0, "removed_fraction": np.nan})
            continue
        threshold = float(valid.quantile(1.0 - UPPER_TAIL_FRACTION))
        high_mask = values > threshold
        filtered = values.mask(high_mask).interpolate(method="linear", limit_direction="both")
        if ROLLING_WINDOW_SAMPLES and int(ROLLING_WINDOW_SAMPLES) > 1:
            filtered = filtered.rolling(int(ROLLING_WINDOW_SAMPLES), center=True, min_periods=1).median()
        out[col] = filtered
        out[f"{col}_fillter_removed_upper_tail"] = high_mask.fillna(False).astype(int)
        out[f"{col}_fillter_threshold_p95"] = threshold
        stats.append({
            "column": col,
            "n_rows": len(out),
            "n_valid": int(valid.shape[0]),
            "threshold": threshold,
            "n_removed": int(high_mask.sum()),
            "removed_fraction": float(high_mask.sum() / max(valid.shape[0], 1)),
        })
    return out, stats


def fillter_one_z_tracking_csv(raw_z_path: Path) -> tuple[Path | None, list[dict[str, object]]]:
    """Read one raw z_tracking.csv and save its filtered copy under result_fillter."""
    raw_z_path = Path(raw_z_path).resolve()
    if not raw_z_path.exists():
        return None, [{"status": "missing_z_tracking_csv", "raw_z_tracking_file": str(raw_z_path)}]
    raw = ka.read_csv_flexible(raw_z_path)
    filtered, stats = fillter_z_tracking_frame(raw)
    rel = _safe_relative_path(raw_z_path, DATA_ROOT)
    if rel.name.lower() == "z_tracking.csv":
        rel = rel.with_name("z_tracking_fillter.csv")
    else:
        rel = rel.with_name(f"{rel.stem}_fillter{rel.suffix}")
    filtered_z_path = (FILTERED_Z_TRACKING_ROOT / rel).resolve()
    assert filtered_z_path.is_relative_to(RESULTS_ROOT), filtered_z_path
    filtered_z_path.parent.mkdir(parents=True, exist_ok=True)
    filtered.to_csv(filtered_z_path, index=False)
    rows = []
    for stat in stats:
        rows.append({
            "status": "ok",
            "filter_tag": FILTER_TAG,
            "raw_z_tracking_file": str(raw_z_path),
            "filtered_z_tracking_file": str(filtered_z_path),
            "source_pair_dir": str(raw_z_path.parent),
            **stat,
        })
    return filtered_z_path, rows


def prepare_filltered_z_tracking_sources(trial_file_manifest: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, Path]]:
    """Filter selected z_tracking files and map raw pair folders to filtered pair folders."""
    rows: list[dict[str, object]] = []
    pair_dir_map: dict[str, Path] = {}
    if trial_file_manifest.empty or "side_video_file" not in trial_file_manifest.columns:
        return pd.DataFrame(), pair_dir_map
    side_files = trial_file_manifest["side_video_file"].dropna().astype(str).drop_duplicates().sort_values()
    for side_file in side_files:
        # `side_video_file` is used only as a path locator for sibling z_tracking.csv.
        # No video file is opened or decoded here.
        raw_pair_dir = Path(side_file).parent.resolve()
        filtered_z_path, stat_rows = fillter_one_z_tracking_csv(raw_pair_dir / "z_tracking.csv")
        if filtered_z_path is not None:
            pair_dir_map[str(raw_pair_dir)] = filtered_z_path.parent
        rows.extend(stat_rows)
    return pd.DataFrame(rows), pair_dir_map


FILTERED_PAIR_DIR_BY_RAW_PAIR_DIR: dict[str, Path] = {}
_ORIGINAL_LOAD_Z_TRACKING_CSV = ka._load_z_tracking_csv
_ORIGINAL_Z_TRACKING_FILENAME = getattr(ka, "Z_TRACKING_FILENAME", "z_tracking.csv")


def load_filltered_z_tracking_csv(side_video: Path) -> pd.DataFrame:
    """Notebook-local side-Z loader: read filtered z_tracking.csv only."""
    raw_pair_dir = Path(side_video).parent.resolve()
    filtered_pair_dir = FILTERED_PAIR_DIR_BY_RAW_PAIR_DIR.get(str(raw_pair_dir))
    if filtered_pair_dir is None:
        expected = FILTERED_Z_TRACKING_ROOT / _safe_relative_path(raw_pair_dir / "z_tracking.csv", DATA_ROOT)
        expected = expected.with_name("z_tracking_fillter.csv")
        return pd.DataFrame({
            "z_tracking_file": [str(expected)],
            "z_tracking_warning": ["missing_filltered_z_tracking_csv"],
            "fillter_tag": [FILTER_TAG],
        })
    # The original loader only looks for sibling z_tracking.csv; the fake mp4 path
    # is never opened. This preserves all existing z_tracking parsing behavior.
    previous_filename = getattr(ka, "Z_TRACKING_FILENAME", "z_tracking.csv")
    ka.Z_TRACKING_FILENAME = "z_tracking_fillter.csv"
    try:
        loaded = _ORIGINAL_LOAD_Z_TRACKING_CSV(Path(filtered_pair_dir) / "side_camera.mp4")
    finally:
        ka.Z_TRACKING_FILENAME = previous_filename
    if isinstance(loaded, pd.DataFrame):
        loaded["fillter_tag"] = FILTER_TAG
        if "side_z_source" in loaded.columns:
            loaded["side_z_source"] = loaded["side_z_source"].astype(str) + f":{FILTER_TAG}"
    return loaded


## 1. Discover trials and filter side-Z CSV inputs

Each selected `answers.csv` row is mapped to `pair_NNN/tracking.csv` and sibling `pair_NNN/z_tracking.csv`.
`side_camera.mp4` path metadata is used only as a folder locator by the existing manifest format; the notebook does not decode video.


In [6]:
trial_file_manifest = ka.discover_trials(
    DATA_ROOT,
    selection=DATA_SELECTION,
    exclude_filter_folders=EXCLUDE_FILTER_FOLDERS,
)
trial_file_manifest["filter_tag"] = FILTER_TAG
ka.save_csv(trial_file_manifest, OUTPUT_ROOT, "trial_file_manifest.csv")
ka.save_experiment_setup_context(OUTPUT_ROOT)

# Clean only result_fillter folders if explicitly requested. Raw data and the
# original analysis/Kinematics/results tree are never touched by this notebook.
_run_root = RUN_OUTPUT_ROOT.resolve()
_selected_subjects = sorted(
    {str(s) for s in trial_file_manifest.get("subject_id", pd.Series(dtype=str)).dropna().unique()}
)
if CLEAR_PREVIOUS_FILTER_OUTPUT:
    _owners = set(_selected_subjects) | set(ka.COMBINED_EXPERIMENT_GROUPS.get(SELECTION_LABEL, ()))
    for _owner in sorted(_owners):
        _owner_dir = (RESULTS_ROOT / ka.sanitize_name(_owner)).resolve()
        assert _owner_dir.is_relative_to(RESULTS_ROOT), _owner_dir
        if _owner_dir == _run_root or _owner_dir in _run_root.parents:
            continue
        ka.clear_output_root(_owner_dir)

z_tracking_filter_manifest, FILTERED_PAIR_DIR_BY_RAW_PAIR_DIR = prepare_filltered_z_tracking_sources(trial_file_manifest)
ka.save_csv(z_tracking_filter_manifest, OUTPUT_ROOT, "z_tracking_fillter_manifest.csv")

# Patch only this notebook's in-memory module. Existing .py/.ipynb files are unchanged.
ka._load_z_tracking_csv = load_filltered_z_tracking_csv

print("Discovered rows:", trial_file_manifest.shape)
print("Selected subjects:", len(_selected_subjects), _selected_subjects)
print("Filtered z_tracking files:", len(FILTERED_PAIR_DIR_BY_RAW_PAIR_DIR))
display(trial_file_manifest.groupby(["subject_group", "selected", "tracking_exists", "side_video_exists"], dropna=False).size().reset_index(name="n"))
display(z_tracking_filter_manifest.head(30))


Discovered rows: (10240, 27)
Selected subjects: 40 ['L_E_1', 'L_E_10', 'L_E_11', 'L_E_12', 'L_E_14', 'L_E_15', 'L_E_16', 'L_E_17', 'L_E_18', 'L_E_2', 'L_E_20', 'L_E_21', 'L_E_22', 'L_E_3', 'L_E_4', 'L_E_5', 'L_E_6', 'L_E_7', 'L_E_8', 'L_E_9', 'N_E_1', 'N_E_11', 'N_E_12', 'N_E_13', 'N_E_14', 'N_E_15', 'N_E_16', 'N_E_17', 'N_E_18', 'N_E_19', 'N_E_2', 'N_E_20', 'N_E_21', 'N_E_3', 'N_E_4', 'N_E_5', 'N_E_6', 'N_E_7', 'N_E_8', 'N_E_9']
Filtered z_tracking files: 10240


,subject_group,selected,tracking_exists,side_video_exists,n
0,L,True,True,True,5120
1,N,True,True,True,5120


,status,filter_tag,raw_z_tracking_file,filtered_z_tracking_file,source_pair_dir,column,n_rows,n_valid,threshold,n_removed,removed_fraction
0,ok,fillter,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,z_active_finger_lift_px,475,475,16.972588,24,0.050526
1,ok,fillter,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,z_hand_midpoint_lift_px,475,0,NaN,0,NaN
2,ok,fillter,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,z_thumb_lift_px,475,0,NaN,0,NaN
3,ok,fillter,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,z_active_finger_lift_px,384,384,16.895755,20,0.052083
4,ok,fillter,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,z_hand_midpoint_lift_px,384,0,NaN,0,NaN
5,ok,fillter,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,z_thumb_lift_px,384,0,NaN,0,NaN
6,ok,fillter,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,z_active_finger_lift_px,392,392,15.140633,20,0.051020
7,ok,fillter,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,z_hand_midpoint_lift_px,392,0,NaN,0,NaN
8,ok,fillter,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,z_thumb_lift_px,392,0,NaN,0,NaN
9,ok,fillter,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,z_active_finger_lift_px,336,336,19.352026,16,0.047619


## 2. XY kinematics from tracking.csv

For every tracking sample the notebook computes location relative to center, polar orientation, LPF/HPF position components, velocity, acceleration, movement direction, radial velocity, and tangential velocity.


In [7]:
kin = ka.compute_tracking_kinematics(
    trial_file_manifest,
    center_x=CENTER_X,
    center_y=CENTER_Y,
    lowpass_cutoff_hz=LOWPASS_CUTOFF_HZ,
    highpass_cutoff_hz=HIGHPASS_CUTOFF_HZ,
    n_time_bins=TRAJECTORY_TIME_BINS,
    max_trials=MAX_TRACKING_TRIALS,
)
kinematic_samples = kin["samples"]
trial_kinematic_summary = kin["trial_summary"]          # one row per participant x pair x stiffness segment
pair_kinematic_summary = kin["pair_summary"]            # one row per recorded pair
trajectory_time_bins = kin["time_bins"]                  # time normalized within each stiffness segment

ka.save_parquet(kinematic_samples, OUTPUT_ROOT, "kinematic_samples.parquet")
ka.save_csv(pair_kinematic_summary, OUTPUT_ROOT, "pair_kinematic_summary.csv")
ka.save_csv(trial_kinematic_summary, OUTPUT_ROOT, "trial_kinematic_summary.csv")
ka.save_csv(trajectory_time_bins, OUTPUT_ROOT, "trajectory_time_bins.csv")
print("samples", kinematic_samples.shape, "stiffness trials", trial_kinematic_summary.shape, "pairs", pair_kinematic_summary.shape, "time bins", trajectory_time_bins.shape)
display(trial_kinematic_summary.head())


samples (3575955, 87) stiffness trials (20480, 108) pairs (10240, 33) time bins (1021379, 86)


,subject_id,subject_group,experiment_group,selected,source_file,run_dir,trial_index_raw,pair_dir,tracking_file,side_video_file,top_video_file,tracking_exists,side_video_exists,pair_number,object_1_finger,object_1_stiffness,object_2_finger,object_2_stiffness,finger_condition,time_to_answer_s,answer_code,comparison_value,standard_value,signed_stiffness_delta,correct_response,warning,filter_tag,tracking_warning,stiffness_value,skin_stretch_gain_mm_per_m,stiffness_segment_id,stiffness_order_in_trial,stiffness_start_time_s,stiffness_end_time_s,stiffness_start_fraction,stiffness_end_fraction,n_tracking_samples,duration_s,pair_duration_s,sampling_rate_hz,mean_x_centered_px,mean_y_centered_px,mean_x_workspace_cm,mean_y_workspace_cm,mean_r_workspace_cm,mean_r_workspace_normalized,workspace_setup,workspace_label,workspace_width_cm,workspace_height_cm,side_camera_side,participant_position_context,movement_space_context,side_camera_interpretation_note,mean_r_center_px,max_r_center_px,max_r_workspace_cm,mean_thumb_active_span_px,mean_thumb_active_span_cm,mean_hand_orientation_xy_deg,path_length_px,path_length_cm,net_displacement_px,net_displacement_cm,straightness_index,straightness_index_cm,mean_vx_px_s,mean_vy_px_s,mean_speed_px_s,max_speed_px_s,mean_vx_cm_s,mean_vy_cm_s,mean_speed_cm_s,max_speed_cm_s,mean_ax_px_s2,mean_ay_px_s2,mean_acceleration_px_s2,max_acceleration_px_s2,mean_ax_cm_s2,mean_ay_cm_s2,mean_acceleration_cm_s2,max_acceleration_cm_s2,mean_jerk_px_s3,max_jerk_px_s3,mean_jerk_cm_s3,max_jerk_cm_s3,normalized_jerk_cost,normalized_jerk_cost_cm,mean_curvature_1_px,median_curvature_1_px,mean_curvature_1_cm,median_curvature_1_cm,speed_curvature_power_law_slope,speed_curvature_power_law_intercept,speed_curvature_power_law_r2,speed_curvature_power_law_n,mean_radial_velocity_px_s,mean_abs_radial_velocity_px_s,mean_abs_tangential_velocity_px_s,mean_radial_velocity_cm_s,mean_abs_radial_velocity_cm_s,mean_abs_tangential_velocity_cm_s,dominant_movement_angle_deg,dominant_movement_angle_cm_deg,dominant_movement_direction,movement_direction_resultant_length,movement_direction_entropy_bits,mean_skin_stretch_gain_mm_per_m
0,L_E_1,L,L_E,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,13,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,True,True,13,M,115.0,M,85.0,M,2.062190,0.0,115.0,85.0,30.0,1.0,,fillter,,115.0,115.0,1,1,0.000000,6.090795,0.000000,0.554307,395,6.090795,10.988135,64.989927,-9.157982,-59.425192,-1.144748,-7.428149,9.352625,0.187052,L,L workspace (80x60 cm),80.0,60.0,right,slightly_left,Lab/airslide 80x60 cm movement field,L/Lab-airslide experiment: top camera was 100 ...,74.820996,162.635709,20.329464,86.773772,10.846721,-56.394513,616.111538,77.013942,0.0,0.0,0.0,0.0,30.972569,30.186605,123.639737,272.542578,3.871571,3.773326,15.454967,34.067822,16.140267,-13.893344,609.489226,3987.902646,2.017533,-1.736668,76.186153,498.487831,48410.531716,446845.502570,6051.316464,55855.687821,1.350577e+08,1.350577e+08,0.011706,0.011706,0.093649,0.093649,-0.414951,2.799818,0.672962,263,0.0,62.351375,28.448091,0.0,7.793922,3.556011,-1.310150,-1.310150,SSE,0.014877,3.055338,115.0
1,L_E_1,L,L_E,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,13,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,True,True,13,M,115.0,M,85.0,M,2.062190,0.0,115.0,85.0,30.0,1.0,,fillter,,85.0,85.0,2,2,6.106709,10.988135,0.555755,1.000000,311,4.881426,10.988135,64.989927,-11.421103,-32.111077,-1.427638,-4.013885,6.179038,0.123581,L,L workspace (80x60 cm),80.0,60.0,right,slightly_left,Lab/airslide 80x60 cm movement field,L/Lab-airslide experiment: top camera wa

## 3. Direction, distance, and success summaries

These tables test whether movement direction, distance from center, speed, or acceleration are associated with selection accuracy.


In [8]:
summaries = ka.summarize_kinematics(trial_kinematic_summary, trajectory_time_bins)
direction_success_summary = summaries["direction_success"]
distance_success_summary = summaries["distance_success"]
subject_kinematic_summary = summaries["subject_summary"]
participant_stiffness_kinematic_summary = summaries["participant_stiffness_summary"]
stiffness_kinematic_summary = summaries["stiffness_summary"]
group_time_summary = summaries["group_time"]
stiffness_time_summary = summaries["stiffness_time"]
kinematic_group_metric_summary = summaries["kinematic_group_metric_summary"]
kinematic_group_condition_metric_summary = summaries["kinematic_group_condition_metric_summary"]
kinematic_within_group_condition_comparisons = summaries["kinematic_within_group_condition_comparisons"]
kinematic_between_group_metric_comparisons = summaries["kinematic_between_group_metric_comparisons"]
kinematic_expanded_scope_metric_summary = summaries.get("kinematic_expanded_scope_metric_summary", pd.DataFrame())
kinematic_expanded_scope_pairwise_mean_differences = summaries.get("kinematic_expanded_scope_pairwise_mean_differences", pd.DataFrame())
kinematic_expanded_scope_status = summaries.get("kinematic_expanded_scope_status", pd.DataFrame())

ka.save_csv(direction_success_summary, OUTPUT_ROOT, "direction_success_summary.csv")
ka.save_csv(distance_success_summary, OUTPUT_ROOT, "distance_success_summary.csv")
ka.save_csv(subject_kinematic_summary, OUTPUT_ROOT, "subject_kinematic_summary.csv")
ka.save_csv(participant_stiffness_kinematic_summary, OUTPUT_ROOT, "participant_stiffness_kinematic_summary.csv")
ka.save_csv(stiffness_kinematic_summary, OUTPUT_ROOT, "stiffness_kinematic_summary.csv")
ka.save_parquet(group_time_summary, OUTPUT_ROOT, "group_time_summary.parquet")
ka.save_csv(stiffness_time_summary, OUTPUT_ROOT, "stiffness_time_summary.csv")
ka.save_csv(kinematic_group_metric_summary, OUTPUT_ROOT, "kinematic_group_metric_summary.csv")
ka.save_csv(kinematic_group_condition_metric_summary, OUTPUT_ROOT, "kinematic_group_condition_metric_summary.csv")
ka.save_csv(kinematic_within_group_condition_comparisons, OUTPUT_ROOT, "kinematic_within_group_condition_comparisons.csv")
ka.save_csv(kinematic_between_group_metric_comparisons, OUTPUT_ROOT, "kinematic_between_group_metric_comparisons.csv")
ka.save_csv(kinematic_expanded_scope_metric_summary, OUTPUT_ROOT, "kinematic_expanded_scope_metric_summary.csv")
ka.save_csv(kinematic_expanded_scope_pairwise_mean_differences, OUTPUT_ROOT, "kinematic_expanded_scope_pairwise_mean_differences.csv")
ka.save_csv(kinematic_expanded_scope_status, OUTPUT_ROOT, "kinematic_expanded_scope_status.csv")

print("Direction x success per participant/stiffness")
display(direction_success_summary.sort_values(["subject_id", "stiffness_value", "finger_condition", "dominant_movement_direction"]).head(40))
print("Participant x stiffness kinematic summary")
display(participant_stiffness_kinematic_summary.head(40))
print("All participants by stiffness")
display(stiffness_kinematic_summary.sort_values("stiffness_value").head(40))

print("N_E / L_E / L_P kinematic group summary")
display(kinematic_group_metric_summary.head(80))
print("Between-group kinematic comparisons")
display(kinematic_between_group_metric_comparisons.head(80))
print("Expanded requested scopes: all, protocol, E/P, N/L, sex, age, stiffness, finger, success/failure")
display(kinematic_expanded_scope_status)
display(kinematic_expanded_scope_metric_summary.head(120))


c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\A

Direction x success per participant/stiffness


,subject_id,subject_group,experiment_group,workspace_setup,side_camera_side,participant_position_context,stiffness_value,finger_condition,dominant_movement_direction,n_trials,success_rate,mean_x_workspace_cm,mean_y_workspace_cm,mean_speed_cm_s,mean_r_workspace_cm,mean_path_length_cm,mean_x_centered_px,mean_y_centered_px,mean_speed_px_s,mean_r_center_px,mean_path_length_px
0,L_E_1,L,L_E,L,right,slightly_left,25.0,I,NNE,1,1.000000,-1.148528,-2.611585,26.629379,12.916837,75.357588,-9.188223,-20.892677,213.035030,103.334700,602.860700
1,L_E_1,L,L_E,L,right,slightly_left,25.0,I,SE,1,1.000000,-1.409438,-7.534572,31.988070,13.601124,84.240510,-11.275504,-60.276575,255.904558,108.808988,673.924076
2,L_E_1,L,L_E,L,right,slightly_left,25.0,I,SW,1,1.000000,-0.880394,-9.066423,31.113887,12.957877,79.824797,-7.043152,-72.531381,248.911098,103.663014,638.598375
3,L_E_1,L,L_E,L,right,slightly_left,25.0,I,W,2,1.000000,-0.257697,-8.157498,32.361963,12.967205,80.764452,-2.061572,-65.259987,258.895702,103.737640,646.115613
4,L_E_1,L,L_E,L,right,slightly_left,25.0,I,WSW,3,1.000000,-1.322667,-5.600028,28.510509,11.540271,72.819713,-10.581337,-44.800222,228.084074,92.322164,582.557700
5,L_E_1,L,L_E,L,right,slightly_left,25.0,M,NNE,1,1.000000,-3.984179,-8.342406,29.307770,13.782894,82.054533,-31.873436,-66.739247,234.462161,110.263150,656.436265
6,L_E_1,L,L_E,L,right,slightly_left,25.0,M,SSW,3,1.000000,1.133361,-7.562040,24.276011,13.087933,127.766697,9.066888,-60.496323,194.208089,104.703462,1022.133577
7,L_E_1,L,L_E,L,right,slightly_left,25.0,M,W,1,1.000000,0.228845,-2.058463,31.382553,12.094898,98.920082,1.830758,-16.467705,251.060428,96.759182,791.360653
8,L_E_1,L,L_E,L,right,slightly_left,25.0,M,WSW,3,1.000000,-1.506433,-5.771032,24.754577,13.270873,86.858813,-12.051464,-46.168256,198.036613,106.166981,694.870502
9,L_E_1,L,L_E,L,right,slightly_left,25.0,P,ESE,1,1.000000,1.319302,-6.272842,12.903469,6.431291,38.940475,10.554419,-50.182736,103.227750,51.450327,311.523798


Participant x stiffness kinematic summary


,subject_id,subject_group,experiment_group,workspace_setup,side_camera_side,participant_position_context,stiffness_value,finger_condition,n_trials,success_rate,mean_x_centered_px,mean_y_centered_px,mean_x_workspace_cm,mean_y_workspace_cm,mean_r_workspace_cm,mean_r_workspace_normalized,mean_max_r_center_px,mean_max_r_workspace_cm,mean_thumb_active_span_px,mean_thumb_active_span_cm,mean_hand_orientation_xy_deg,mean_hand_orientation_xy_cm_deg,mean_vx_px_s,mean_vy_px_s,mean_vx_cm_s,mean_vy_cm_s,mean_speed_px_s,mean_speed_cm_s,mean_ax_px_s2,mean_ay_px_s2,mean_ax_cm_s2,mean_ay_cm_s2,mean_acceleration_px_s2,mean_acceleration_cm_s2,mean_jerk_px_s3,mean_jerk_cm_s3,mean_normalized_jerk_cost,mean_normalized_jerk_cost_cm,mean_curvature_1_px,mean_curvature_1_cm,speed_curvature_power_law_slope,speed_curvature_power_law_r2,mean_path_length_px,mean_path_length_cm,mean_straightness_index,mean_straightness_index_cm,circular_mean_direction_deg,movement_direction_resultant_length
0,L_E_1,L,L_E,L,right,slightly_left,25.0,I,8,1.000000,-9.189720,-46.493343,-1.148715,-5.811668,12.937357,0.258747,176.218282,22.027285,65.999405,8.249926,-63.955510,NaN,45.216947,22.742699,5.652118,2.842837,245.026076,30.628260,55.741901,-5.359383,6.967738,-0.669923,1408.509583,176.063698,98607.300439,12325.912555,2.825172e+07,2.825172e+07,0.005753,0.046021,-0.328050,0.577481,620.729538,77.591192,0.000000,0.000000,-148.569588,0.686528
1,L_E_1,L,L_E,L,right,slightly_left,25.0,M,8,1.000000,-11.953369,-59.279457,-1.494171,-7.409932,13.179403,0.263588,212.316570,26.539571,82.316317,10.289540,-57.223924,NaN,38.009610,18.238002,4.751201,2.279750,216.249387,27.031173,47.298334,-46.777393,5.912292,-5.847174,1245.358128,155.669766,85776.813714,10722.101714,4.258448e+07,4.258448e+07,0.007779,0.062228,-0.345427,0.616404,699.243962,87.405495,0.000000,0.000000,-120.215714,0.767906
2,L_E_1,L,L_E,L,right,slightly_left,25.0,P,8,1.000000,1.238934,-49.728137,0.154867,-6.216017,10.538953,0.210779,171.594919,21.449365,106.776966,13.347121,-22.092501,NaN,0.443684,2.521100,0.055461,0.315137,134.665496,16.833187,-2.484058,-28.017591,-0.310507,-3.502199,774.374449,96.796806,61141.446603,7642.680825,4.737381e+07,4.737381e+07,0.007073,0.056580,-0.321488,0.626014,525.024661,65.628083,0.000000,0.000000,-43.961428,0.372481
3,L_E_1,L,L_E,L,right,slightly_left,25.0,R,8,1.000000,93.802452,-2.248256,11.725307,-0.281032,11.774526,0.235491,204.445885,25.555736,91.809184,11.476148,-27.440583,NaN,7.384698,0.052017,0.923087,0.006502,115.325174,14.415647,-4.191735,7.465567,-0.523967,0.933196,619.471306,77.433913,30304.281254,3788.035157,1.696625e+08,1.696625e+08,0.004180,0.033436,-0.417535,0.796598,426.603330,53.325416,0.000000,0.000000,-84.595068,0.095991
4,L_E_1,L,L_E,L,right,slightly_left,40.0,I,8,1.000000,8.698692,-63.319863,1.087336,-7.914983,13.680006,0.273600,192.351474,24.043934,63.782038,7.972755,-63.028480,NaN,15.573473,10.244637,1.946684,1.280580,254.109396,31.763674,5.011175,-3.969108,0.626397,-0.496139,1539.124095,192.390512,112686.688417,14085.836052,3.475786e+07,3.475786e+07,0.005249,0.041992,-0.353914,0.603494,675.005341,84.375668,0.000000,0.000000,-45.994214,0.806926
5,L_E_1,L,L_E,L,right,slightly_left,40.0,M,8,1.000000,-11.885674,-57.785693,-1.485709,-7.223212,12.387426,0.247749,181.227490,22.653436,83.443053,10.430382,-57.770629,NaN,42.573539,17.638831,5.321692,2.204854,199.681181,24.960148,32.769237,21.341882,4.096155,2.667735,1187.174238,148.396780,87918.214213,10989.776777,7.966027e+07,7.966027e+07,0.006705,0.053644,-0.394767,0.591016,639.251044,79.906380,0.000000,0.000000,-37.642225,0.560095
6,L_E_1,L,L_E,L,right,slightly_left,40.0,P,8,0.875000,-2.746983,-21.506703,-0.343373,-2.688338,8.770042,0.175401,193.106741,24.138343,110.793637,13.849205,-20.735187,NaN,-0.622391,0.963665,-0.077799,0.120458,145.434998,18.179375,-5.255981,-14.198003,-0.656998,-1.774750,717.236824,89.654603,59400.214786,7425.026848,4.289162e+07,4.289162e+07,0.006106,0.048845,-0.341044,0.666640,506.030033,63.253754,0.

...

Between-group kinematic comparisons


,condition_col,condition_level,group_a,group_b,comparison,metric,n_subjects_a,n_subjects_b,mean_a,mean_b,mean_difference_b_minus_a,cohens_d_b_minus_a
0,all,all,N_E,L_E,L_E - N_E,success_rate,20,20,8.130859e-01,7.849609e-01,-2.812500e-02,-0.319073
1,all,all,N_E,L_E,L_E - N_E,mean_x_workspace_cm,20,20,1.554821e-01,2.259060e+00,2.103578e+00,0.689816
2,all,all,N_E,L_E,L_E - N_E,mean_y_workspace_cm,20,20,-3.151004e-01,-2.264408e-01,8.865953e-02,0.033068
3,all,all,N_E,L_E,L_E - N_E,mean_r_workspace_cm,20,20,5.365472e+00,8.798463e+00,3.432992e+00,1.764405
4,all,all,N_E,L_E,L_E - N_E,mean_r_workspace_normalized,20,20,1.430793e-01,1.759693e-01,3.289002e-02,0.773951
...,...,...,...,...,...,...,...,...,...,...,...,...
75,finger_condition,P,N_E,L_E,L_E - N_E,mean_acceleration_cm_s2,20,20,1.155621e+02,1.171631e+02,1.600991e+00,0.027248
76,finger_condition,P,N_E,L_E,L_E - N_E,mean_jerk_cm_s3,20,20,2.530620e+03,5.424147e+03,2.893527e+03,0.917540
77,finger_condition,P,N_E,L_E,L_E - N_E,mean_normalized_jerk_cost_cm,20,20,1.145014e+07,1.110945e+09,1.099495e+09,0.351777
78,finger_condition,P,N_E,L_E,L_E - N_E,mean_curvature_1_cm,20,20,8.233624e-02,4.072027e-02,-4.161597e-02,-0.941970


Expanded requested scopes: all, protocol, E/P, N/L, sex, age, stiffness, finger, success/failure


,comparison_scope,status,column,note,n_levels
0,all,available,all,all_participants,1
1,protocol,missing,protocol_factor,protocol 1/2/3/4 if present/inferred,0
2,E_vs_P,available,subject_group,E vs P,2
3,N_vs_L_workspace,available,workspace_setup,N=60x45 cm vs L=80x60 cm,2
4,side_camera,available,side_camera_side,right-side L vs left-side N camera,2
5,experiment_group,available,experiment_group,N_E/L_E/L_P exact group,2
6,sex,missing,sex_factor,male vs female if present,0
7,age,missing,age_group,age bins if present,0
8,stiffness,available,stiffness_value,skin-stretch/stiffness value,9
9,finger,available,finger_condition,active finger,4


,comparison_scope,metric,comparison_value,n_observations,n_subjects,mean,median,sem,ci95_mean_low,ci95_mean_high
0,all,success_rate,all_participants,1422,40,0.797617,0.859375,0.004795,0.788219,0.807015
1,all,mean_x_workspace_cm,all_participants,1422,40,1.156396,0.348356,0.095526,0.969166,1.343626
2,all,mean_y_workspace_cm,all_participants,1422,40,-0.270794,-0.099038,0.076571,-0.420873,-0.120715
3,all,mean_r_workspace_cm,all_participants,1422,40,7.068480,6.537000,0.075209,6.921070,7.215889
4,all,mean_r_workspace_normalized,all_participants,1422,40,0.159164,0.152099,0.001378,0.156464,0.161865
...,...,...,...,...,...,...,...,...,...,...
115,side_camera,mean_y_workspace_cm,right,711,20,-0.224520,-0.179954,0.136981,-0.493004,0.043964
116,side_camera,mean_r_workspace_cm,left,711,20,5.338457,5.309537,0.059910,5.221033,5.455881
117,side_camera,mean_r_workspace_cm,right,711,20,8.798502,8.214929,0.103059,8.596506,9.000498
118,side_camera,mean_r_workspace_normalized,left,711,20,0.142359,0.141588,0.001598,0.139228,0.145490


## 4. Motor-control repeated-measures calculations

These calculations avoid trial-pooling across people. They produce: within-subject centered/z-scored kinematics, within-finger stiffness sensitivities for each subject, and paired finger comparisons using only subjects who contributed both fingers in a contrast. This is the preferred structure for motor-control interpretation.


In [9]:
motor_control = ka.compute_motor_control_comparisons(subject_kinematic_summary)
kinematic_within_subject = motor_control["within_subject"]
finger_metric_summary = motor_control["finger_metric_summary"]
within_finger_stiffness_effects = motor_control["within_finger_stiffness_effects"]
within_finger_stiffness_effect_summary = motor_control["within_finger_stiffness_effect_summary"]
finger_comparison_paired = motor_control["finger_comparison_paired"]
finger_comparison_by_stiffness_paired = motor_control["finger_comparison_by_stiffness_paired"]

ka.save_csv(kinematic_within_subject, OUTPUT_ROOT, "kinematic_within_subject.csv")
ka.save_csv(finger_metric_summary, OUTPUT_ROOT, "finger_metric_summary.csv")
ka.save_csv(within_finger_stiffness_effects, OUTPUT_ROOT, "within_finger_stiffness_effects.csv")
ka.save_csv(within_finger_stiffness_effect_summary, OUTPUT_ROOT, "within_finger_stiffness_effect_summary.csv")
ka.save_csv(finger_comparison_paired, OUTPUT_ROOT, "finger_comparison_paired.csv")
ka.save_csv(finger_comparison_by_stiffness_paired, OUTPUT_ROOT, "finger_comparison_by_stiffness_paired.csv")

motor_control_figure_paths = ka.save_motor_control_figures(OUTPUT_ROOT, motor_control, fig_dpi=FIG_DPI)

print("Within-subject rows:", kinematic_within_subject.shape)
display(kinematic_within_subject.head(30))
print("Within-finger stiffness effects: one slope per subject x finger x metric")
display(within_finger_stiffness_effects.head(30))
print("Within-finger stiffness-effect summary")
display(within_finger_stiffness_effect_summary.sort_values(["metric", "finger_condition"]).head(80))
print("Paired finger comparisons (finger_b - finger_a)")
display(finger_comparison_paired.sort_values(["metric", "comparison"]).head(80))
print(f"Saved {len(motor_control_figure_paths)} motor-control figures")


Within-subject rows: (1422, 57)


,subject_id,subject_group,finger_condition,stiffness_value,n_trials,success_rate,mean_max_r_workspace_cm,mean_speed_cm_s,mean_acceleration_cm_s2,mean_jerk_cm_s3,mean_normalized_jerk_cost_cm,mean_curvature_1_cm,speed_curvature_power_law_slope,speed_curvature_power_law_r2,mean_path_length_cm,mean_straightness_index_cm,mean_vx_cm_s,mean_vy_cm_s,success_rate_within_subject_centered,success_rate_within_subject_z,success_rate_within_subject_finger_centered,mean_max_r_workspace_cm_within_subject_centered,mean_max_r_workspace_cm_within_subject_z,mean_max_r_workspace_cm_within_subject_finger_centered,mean_speed_cm_s_within_subject_centered,mean_speed_cm_s_within_subject_z,mean_speed_cm_s_within_subject_finger_centered,mean_acceleration_cm_s2_within_subject_centered,mean_acceleration_cm_s2_within_subject_z,mean_acceleration_cm_s2_within_subject_finger_centered,mean_jerk_cm_s3_within_subject_centered,mean_jerk_cm_s3_within_subject_z,mean_jerk_cm_s3_within_subject_finger_centered,mean_normalized_jerk_cost_cm_within_subject_centered,mean_normalized_jerk_cost_cm_within_subject_z,mean_normalized_jerk_cost_cm_within_subject_finger_centered,mean_curvature_1_cm_within_subject_centered,mean_curvature_1_cm_within_subject_z,mean_curvature_1_cm_within_subject_finger_centered,speed_curvature_power_law_slope_within_subject_centered,speed_curvature_power_law_slope_within_subject_z,speed_curvature_power_law_slope_within_subject_finger_centered,speed_curvature_power_law_r2_within_subject_centered,speed_curvature_power_law_r2_within_subject_z,speed_curvature_power_law_r2_within_subject_finger_centered,mean_path_length_cm_within_subject_centered,mean_path_length_cm_within_subject_z,mean_path_length_cm_within_subject_finger_centered,mean_straightness_index_cm_within_subject_centered,mean_straightness_index_cm_within_subject_z,mean_straightness_index_cm_within_subject_finger_centered,mean_vx_cm_s_within_subject_centered,mean_vx_cm_s_within_subject_z,mean_vx_cm_s_within_subject_finger_centered,mean_vy_cm_s_within_subject_centered,mean_vy_cm_s_within_subject_z,mean_vy_cm_s_within_subject_finger_centered
0,L_E_1,L,I,25.0,8,1.000000,22.027285,30.628260,176.063698,12325.912555,2.825172e+07,0.046021,-0.328050,0.577481,77.591192,0.000000,5.652118,2.842837,0.187500,1.052007,0.109375,-2.529635,-1.319190,-2.031994,8.788520,1.326795,0.641558,55.597136,1.259023,-3.643763,3719.296248,1.021089,-629.131834,-1.382785e+08,-1.148165,-1.324761e+08,0.001757,0.138933,0.009311,0.044380,1.340263,0.029015,-0.102409,-1.454906,-0.058192,-7.229544,-0.242413,-36.563108,-0.000344,-0.238749,0.000000,4.084791,2.254618,4.076520,2.364135,2.992235,2.165762
1,L_E_1,L,M,25.0,8,1.000000,26.539571,27.031173,155.669766,10722.101714,4.258448e+07,0.062228,-0.345427,0.616404,87.405495,0.000000,4.751201,2.279750,0.187500,1.052007,0.140625,1.982651,1.033941,1.038387,5.191434,0.783746,1.198696,35.203204,0.797193,13.306210,2115.485407,0.580782,63.076079,-1.239457e+08,-1.029156,-1.230840e+08,0.017965,1.420279,0.011793,0.027003,0.815487,0.013210,-0.063487,-0.901941,-0.007897,2.584759,0.086669,-17.345240,-0.000344,-0.238749,0.000000,3.183873,1.757353,1.140069,1.801048,2.279548,1.131966
2,L_E_1,L,P,25.0,8,1.000000,21.449365,16.833187,96.796806,7642.680825,4.737381e+07,0.056580,-0.321488,0.626014,65.628083,0.000000,0.055461,0.315137,0.187500,1.052007,0.218750,-3.107556,-1.620572,-1.116080,-5.006552,-0.755835,-0.752858,-23.669756,-0.536013,11.472469,-963.935482,-0.264637,410.099863,-1.191564e+08,-0.989389,-3.584032e+07,0.012317,0.973769,-0.001953,0.050942,1.538407,0.035393,-0.053876,-0.765411,-0.047147,-19.192654,-0.643547,-0.815490,-0.000344,-0.238749,0.000000,-1.511867,-0.834482,-0.373964,-0.163565,-0.207020,0.312485
3,L_E_1,L,R,25.0,8,1.000000,25.555736,14.415647,77.433913,3788.035157,1.696625e+08,0.033436,-0.417535,0.796598,53.325416,0.000000,0.923087,0.006502,0.187500,1.052007,0.281250,0.998815,0.520876,-0.546038,-7.424092,-1.120808,0.461914,-43.032649,-0.974495,2.963020,-4818.581150,-1.322885,208.22091

Within-finger stiffness effects: one slope per subject x finger x metric


,subject_id,finger_condition,metric,n_stiffness_levels,slope_per_stiffness_unit,high_minus_low_stiffness_delta
0,L_E_1,I,success_rate,9,-9.722222e-04,0.000000e+00
1,L_E_1,I,mean_max_r_workspace_cm,9,1.816333e-02,2.174922e+00
2,L_E_1,I,mean_speed_cm_s,9,-3.321859e-03,-2.132738e+00
3,L_E_1,I,mean_acceleration_cm_s2,9,-5.160690e-02,-1.168756e+01
4,L_E_1,I,mean_jerk_cm_s3,9,3.022258e+00,1.078315e+03
5,L_E_1,I,mean_normalized_jerk_cost_cm,9,2.155855e+06,5.098444e+07
6,L_E_1,I,mean_curvature_1_cm,9,-1.194898e-04,-8.591185e-03
7,L_E_1,I,speed_curvature_power_law_slope,9,-8.462838e-05,-6.814299e-03
8,L_E_1,I,speed_curvature_power_law_r2,9,4.308128e-04,8.976702e-02
9,L_E_1,I,mean_path_length_cm,9,5.125095e-01,4.486400e+01


Within-finger stiffness-effect summary


,finger_condition,metric,n_subjects_with_slope,mean_slope_per_stiffness_unit,sem_slope_per_stiffness_unit,slope_sign_flip_p,n_subjects_with_high_low_delta,mean_high_minus_low_stiffness_delta,sem_high_minus_low_stiffness_delta,high_low_delta_sign_flip_p
0,I,mean_acceleration_cm_s2,40,2.264426e-03,2.877795e-02,0.940403,40,1.761378e+00,5.541204e+00,0.781411
13,M,mean_acceleration_cm_s2,39,-2.271199e-02,2.709939e-02,0.425929,39,-2.426666e+00,4.230144e+00,0.575671
26,P,mean_acceleration_cm_s2,40,-1.991167e-02,1.520421e-02,0.196740,40,-8.654482e-01,3.367834e+00,0.799760
39,R,mean_acceleration_cm_s2,39,-1.450471e-02,2.345628e-02,0.561722,39,-1.226417e+00,4.394005e+00,0.800460
1,I,mean_curvature_1_cm,40,8.021358e-05,3.729624e-05,0.012449,40,9.751261e-03,6.507851e-03,0.100645
14,M,mean_curvature_1_cm,39,1.027356e-04,4.445118e-05,0.015349,39,9.937004e-03,5.949487e-03,0.105595
27,P,mean_curvature_1_cm,40,4.540680e-05,2.596774e-05,0.034748,40,9.125470e-03,4.887121e-03,0.022699
40,R,mean_curvature_1_cm,39,1.505099e-05,2.949123e-05,0.631568,39,5.341289e-03,4.030432e-03,0.197440
2,I,mean_jerk_cm_s3,40,-9.092321e-01,8.039070e-01,0.272936,40,-1.251899e+02,1.988083e+02,0.563722
15,M,mean_jerk_cm_s3,39,-5.534218e-01,1.001174e+00,0.594720,39,-2.642019e+01,1.251585e+02,0.835858


Paired finger comparisons (finger_b - finger_a)


,finger_a,finger_b,comparison,metric,mean_a,mean_b,n_paired_observations,mean_difference,median_difference,sem_difference,cohens_dz,sign_flip_p
3,I,M,M - I,mean_acceleration_cm_s2,136.248085,143.340073,39,7.091989,0.645728,7.504822,0.151320,0.354982
29,I,P,P - I,mean_acceleration_cm_s2,134.210572,116.362634,40,-17.847938,-0.916559,7.471787,-0.377688,0.022399
55,M,P,P - M,mean_acceleration_cm_s2,143.340073,117.866312,39,-25.473761,-9.349790,11.420592,-0.357168,0.028099
68,R,P,P - R,mean_acceleration_cm_s2,138.519528,117.661199,39,-20.858329,-4.957362,9.928952,-0.336391,0.039398
16,I,R,R - I,mean_acceleration_cm_s2,135.210882,138.519528,39,3.308645,8.028152,8.096470,0.065437,0.685766
...,...,...,...,...,...,...,...,...,...,...,...,...
26,I,P,P - I,success_rate,0.797266,0.807617,40,0.010352,0.023438,0.011103,0.147414,0.362932
52,M,P,P - M,success_rate,0.778045,0.804087,39,0.026042,0.015625,0.013367,0.311955,0.060947
65,R,P,P - R,success_rate,0.807292,0.805889,39,-0.001402,0.000000,0.009440,-0.023786,0.897605
13,I,R,R - I,success_rate,0.796074,0.807292,39,0.011218,0.000000,0.012270,0.146398,0.384331


Saved 3 motor-control figures


## 5. Side-camera Z/lift estimate

This filtered notebook uses the filtered `z_tracking.csv` side-view tracking copies created under `result_fillter`; it does not decode side-camera video. This section reports both the raw side-camera pixel proxy and a setup-scaled centimeter proxy. The cm proxy uses the same setup-specific vertical scale as the top-camera movement field because no independent side-camera calibration grid is stored with the analysis. Positive values mean the hand/finger region is higher in the side image relative to the within-trial baseline. Camera placement is normalized by setup: `L` side camera on the right, `N` side camera on the left, both 10 cm above the table.


In [10]:
side = ka.estimate_side_video_z(
    trial_kinematic_summary,
    samples_per_video=SIDE_VIDEO_SAMPLES_PER_TRIAL,
    max_trials=MAX_SIDE_VIDEO_TRIALS,
)
side_z_samples = side["side_samples"]
side_z_trial_summary = side["side_trial_summary"]
side_z_group_time_summary = side["side_group_time"]
side_z_subject_stiffness_summary = side["side_subject_stiffness_summary"]
side_z_by_stiffness_summary = side["side_stiffness_summary"]

ka.save_parquet(side_z_samples, OUTPUT_ROOT, "side_z_samples.parquet")
ka.save_csv(side_z_trial_summary, OUTPUT_ROOT, "side_z_trial_summary.csv")
ka.save_csv(side_z_group_time_summary, OUTPUT_ROOT, "side_z_group_time_summary.csv")
ka.save_csv(side_z_subject_stiffness_summary, OUTPUT_ROOT, "side_z_subject_stiffness_summary.csv")
ka.save_csv(side_z_by_stiffness_summary, OUTPUT_ROOT, "side_z_by_stiffness_summary.csv")

print("side samples", side_z_samples.shape, "side stiffness trial summary", side_z_trial_summary.shape)
display(side_z_trial_summary.head())
print("Z/lift by stiffness")
display(side_z_by_stiffness_summary.sort_values("stiffness_value").head(30))


side samples (614382, 176) side stiffness trial summary (20480, 46)


,subject_id,trial_index_raw,stiffness_segment_id,subject_group,experiment_group,finger_condition,stiffness_value,stiffness_order_in_trial,comparison_value,standard_value,signed_stiffness_delta,correct_response,side_video_file,z_tracking_file,z_tracking_warning,missing_z_tracking_csv,side_camera_side,side_camera_view_sign,n_side_samples,side_detection_rate,mean_side_z_lift_px,max_side_z_lift_px,mean_side_x_from_center_raw_px,mean_side_x_from_center_camera_corrected_px,max_abs_side_x_from_center_camera_corrected_px,mean_side_lift_lateral_angle_raw_deg,mean_side_lift_lateral_angle_camera_corrected_deg,mean_side_motion_direction_camera_corrected_deg,mean_side_mask_area_px,workspace_setup,workspace_label,workspace_width_cm,workspace_height_cm,top_camera_height_above_table_cm,side_camera_height_above_table_cm,top_camera_frame_width_px,top_camera_frame_height_px,x_cm_per_px,y_cm_per_px,side_z_cm_per_px,participant_position_context,movement_space_context,side_camera_interpretation_note,mean_side_x_from_center_camera_corrected_cm,mean_side_z_lift_cm,max_side_z_lift_cm
0,L_E_1,13,1,L,L_E,M,115.0,1,115.0,85.0,30.0,1.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,,0,right,1.0,30,1.0,12.098347,29.276152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,L,L workspace (80x60 cm),80.0,60.0,100.0,10.0,640.0,480.0,0.125,0.125,0.125,slightly_left,Lab/airslide 80x60 cm movement field,L/Lab-airslide experiment: top camera was 100 ...,NaN,1.512293,3.659519
1,L_E_1,13,2,L,L_E,M,85.0,2,115.0,85.0,30.0,1.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,,0,right,1.0,30,1.0,20.063706,29.974700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,L,L workspace (80x60 cm),80.0,60.0,100.0,10.0,640.0,480.0,0.125,0.125,0.125,slightly_left,Lab/airslide 80x60 cm movement field,L/Lab-airslide experiment: top camera was 100 ...,NaN,2.507963,3.746838
2,L_E_1,14,1,L,L_E,M,55.0,1,55.0,85.0,-30.0,1.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,,0,right,1.0,30,1.0,16.924711,30.199740,NaN,NaN,NaN,NaN,NaN,NaN,NaN,L,L workspace (80x60 cm),80.0,60.0,100.0,10.0,640.0,480.0,0.125,0.125,0.125,slightly_left,Lab/airslide 80x60 cm movement field,L/Lab-airslide experiment: top camera was 100 ...,NaN,2.115589,3.774968
3,L_E_1,14,2,L,L_E,M,85.0,2,55.0,85.0,-30.0,1.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,,0,right,1.0,30,1.0,11.381583,27.014313,NaN,NaN,NaN,NaN,NaN,NaN,NaN,L,L workspace (80x60 cm),80.0,60.0,100.0,10.0,640.0,480.0,0.125,0.125,0.125,slightly_left,Lab/airslide 80x60 cm movement field,L/Lab-airslide experiment: top camera was 100 ...,NaN,1.422698,3.376789
4,L_E_1,15,1,L,L_E,M,40.0,1,40.0,85.0,-45.0,1.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,,0,right,1.0,30,1.0,19.666888,37.970260,NaN,NaN,NaN,NaN,NaN,NaN,NaN,L,L workspace (80x60 cm),80.0,60.0,100.0,10.0,640.0,480.0,0.125,0.125,0.125,slightly_left,Lab/airslide 80x60 cm movement field,L/Lab-airslide experiment: top camera was 100 ...,NaN,2.458361,4.746283


Z/lift by stiffness


,stiffness_value,n_trials,n_subjects,mean_side_z_lift_px,sem_side_z_lift_px,max_side_z_lift_px,mean_side_z_lift_cm,sem_side_z_lift_cm,max_side_z_lift_cm,mean_side_x_from_center_camera_corrected_px,mean_side_x_from_center_camera_corrected_cm,mean_side_lift_lateral_angle_camera_corrected_deg,mean_side_motion_direction_camera_corrected_deg,success_rate
0,25.0,1280,40,38.368488,1.177585,84.720481,4.136367,0.136296,9.062813,NaN,NaN,NaN,NaN,0.955469
1,40.0,1280,40,37.725843,1.139364,84.301463,4.021847,0.132336,9.023413,NaN,NaN,NaN,NaN,0.911719
2,55.0,1280,40,40.363465,1.177561,86.367046,4.363461,0.135604,9.218986,NaN,NaN,NaN,NaN,0.838281
3,70.0,1280,40,37.646160,1.222182,87.242911,3.956210,0.140549,9.292168,NaN,NaN,NaN,NaN,0.676562
4,85.0,10240,40,38.366988,0.428003,86.075205,4.113060,0.050119,9.202312,NaN,NaN,NaN,NaN,0.799023
5,100.0,1280,40,39.929404,1.188631,85.682582,4.131185,0.140273,9.166716,NaN,NaN,NaN,NaN,0.667969
6,115.0,1280,40,38.298598,1.288417,85.951323,4.059427,0.153754,9.190282,NaN,NaN,NaN,NaN,0.742969
7,130.0,1280,40,37.479655,1.287534,83.312978,4.001124,0.152892,8.923941,NaN,NaN,NaN,NaN,0.782813
8,145.0,1280,40,38.531469,1.242964,83.722871,4.001472,0.146375,8.936857,NaN,NaN,NaN,NaN,0.816406


In [ ]:
# --- 5b. Z-lift cm L/N colors and baseline-corrected trajectory ---
# L is pink, N is purple. This plot uses centimeters only and subtracts each
# segment's first-10-sample baseline to show lift relative to the trial start.
import matplotlib.pyplot as plt

Z_LIFT_GROUP_COLORS = {
    "L_E": "#FF69B4",  # pink
    "N_E": "#7B2CBF",  # purple
}

def _z_lift_sem(values: pd.Series) -> float:
    values = pd.to_numeric(values, errors="coerce").dropna()
    return float(values.std(ddof=1) / np.sqrt(values.size)) if values.size > 1 else np.nan

z_lift_cm_timecourse = pd.DataFrame()
z_lift_plot_path = None

if side_z_samples.empty or "side_z_lift_cm" not in side_z_samples.columns:
    print("No side_z_lift_cm samples available for the L/N cm Z-lift plot.")
else:
    z_lift_plot = side_z_samples.copy()
    if ka.EXPERIMENT_GROUP_COLUMN not in z_lift_plot.columns:
        z_lift_plot[ka.EXPERIMENT_GROUP_COLUMN] = (
            z_lift_plot["subject_id"].astype(str).str.extract(r"^(L_E|N_E)", expand=False)
        )

    z_lift_plot = z_lift_plot[
        z_lift_plot[ka.EXPERIMENT_GROUP_COLUMN].isin(Z_LIFT_GROUP_COLORS)
    ].copy()

    if z_lift_plot.empty:
        print("No L_E/N_E rows available for the L/N cm Z-lift plot.")
    else:
        z_lift_keys = ["subject_id", "trial_index_raw", "stiffness_segment_id"]
        for col in ["trial_index_raw", "stiffness_segment_id", "side_time_fraction", "side_time_bin", "side_z_lift_cm"]:
            z_lift_plot[col] = pd.to_numeric(z_lift_plot[col], errors="coerce")

        z_lift_plot = z_lift_plot.dropna(
            subset=[ka.EXPERIMENT_GROUP_COLUMN, "side_z_lift_cm", "side_time_fraction", "side_time_bin"]
        ).sort_values(z_lift_keys + ["side_time_fraction", "side_time_bin"])

        z_lift_plot["_baseline_rank"] = z_lift_plot.groupby(z_lift_keys).cumcount()
        z_lift_baseline = (
            z_lift_plot[z_lift_plot["_baseline_rank"] < 10]
            .groupby(z_lift_keys, as_index=False)
            .agg(z_lift_baseline_cm=("side_z_lift_cm", "mean"), z_lift_baseline_n=("side_z_lift_cm", "count"))
        )
        z_lift_plot = z_lift_plot.merge(z_lift_baseline, on=z_lift_keys, how="left")
        z_lift_plot["side_z_lift_delta_from_first10_cm"] = (
            z_lift_plot["side_z_lift_cm"] - z_lift_plot["z_lift_baseline_cm"]
        )

        z_lift_cm_timecourse = (
            z_lift_plot.groupby(["side_time_bin", ka.EXPERIMENT_GROUP_COLUMN], as_index=False)
            .agg(
                side_time_fraction=("side_time_fraction", "mean"),
                abs_cm=("side_z_lift_cm", "mean"),
                abs_sem_cm=("side_z_lift_cm", _z_lift_sem),
                delta_cm=("side_z_lift_delta_from_first10_cm", "mean"),
                delta_sem_cm=("side_z_lift_delta_from_first10_cm", _z_lift_sem),
                n_samples=("side_z_lift_cm", "size"),
            )
            .sort_values(["side_time_bin", ka.EXPERIMENT_GROUP_COLUMN])
        )

        fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
        for group in ["L_E", "N_E"]:
            g = z_lift_cm_timecourse[
                z_lift_cm_timecourse[ka.EXPERIMENT_GROUP_COLUMN].astype(str).eq(group)
            ].sort_values("side_time_bin")
            if g.empty:
                continue
            color = Z_LIFT_GROUP_COLORS[group]
            axes[0].plot(
                g["side_time_fraction"], g["abs_cm"], marker="o", label=group, color=color, linewidth=2.2
            )
            axes[0].fill_between(
                g["side_time_fraction"], g["abs_cm"] - g["abs_sem_cm"], g["abs_cm"] + g["abs_sem_cm"],
                color=color, alpha=0.14
            )
            axes[1].plot(
                g["side_time_fraction"], g["delta_cm"], marker="o", label=group, color=color, linewidth=2.2
            )
            axes[1].fill_between(
                g["side_time_fraction"], g["delta_cm"] - g["delta_sem_cm"], g["delta_cm"] + g["delta_sem_cm"],
                color=color, alpha=0.14
            )

        axes[0].set_title("Absolute side Z/lift (cm)")
        axes[0].set_ylabel("Z/lift (cm)")
        axes[1].set_title("Baseline-corrected side Z/lift (cm)\nsubtract first 10 samples per segment")
        axes[1].set_ylabel("Delta from first-10 baseline (cm)")
        for ax in axes:
            ax.set_xlabel("Normalized side-video time")
            ax.grid(alpha=0.25)
            ax.legend(title="Group")
        fig.tight_layout()

        z_lift_fig_dir = OUTPUT_ROOT / "figures"
        z_lift_fig_dir.mkdir(parents=True, exist_ok=True)
        z_lift_plot_path = z_lift_fig_dir / "side_z_lift_cm_absolute_vs_first10_baseline_by_experiment_group.png"
        fig.savefig(z_lift_plot_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.show()

        ka.save_csv(
            z_lift_cm_timecourse,
            OUTPUT_ROOT,
            "side_z_lift_cm_absolute_vs_first10_baseline_timecourse.csv",
        )
        print("Saved", z_lift_plot_path)
        display(z_lift_cm_timecourse.head(10))



## 6. Advanced success, Z-height, and trajectory structure

IEEE-style motor-control layer: successful-vs-unsuccessful kinematic/Z contrasts are computed within subject and finger; trajectory templates are compared between fingers and between correct/incorrect trials using normalized XY paths.


In [11]:
success_kinematic_z = ka.compute_success_kinematic_z_analysis(trial_kinematic_summary, side_z_trial_summary)
trial_success_kinematic_z_table = success_kinematic_z["trial_success_table"]
success_kinematic_z_contrast_by_subject_finger = success_kinematic_z["success_contrast_by_subject_finger"]
success_kinematic_z_contrast_summary = success_kinematic_z["success_contrast_summary"]
success_kinematic_z_contrast_by_finger_summary = success_kinematic_z["success_contrast_by_finger_summary"]

trajectory_structure = ka.compute_trajectory_similarity_analysis(trajectory_time_bins)
subject_finger_trajectory = trajectory_structure["subject_finger_trajectory"]
trajectory_variability_summary = trajectory_structure["trajectory_variability_summary"]
finger_trajectory_distance_paired = trajectory_structure["finger_trajectory_distance_paired"]
finger_trajectory_distance_summary = trajectory_structure["finger_trajectory_distance_summary"]
success_failure_trajectory_distance = trajectory_structure["success_failure_trajectory_distance"]
success_failure_trajectory_distance_summary = trajectory_structure["success_failure_trajectory_distance_summary"]

ka.save_csv(trial_success_kinematic_z_table, OUTPUT_ROOT, "trial_success_kinematic_z_table.csv")
ka.save_csv(success_kinematic_z_contrast_by_subject_finger, OUTPUT_ROOT, "success_kinematic_z_contrast_by_subject_finger.csv")
ka.save_csv(success_kinematic_z_contrast_summary, OUTPUT_ROOT, "success_kinematic_z_contrast_summary.csv")
ka.save_csv(success_kinematic_z_contrast_by_finger_summary, OUTPUT_ROOT, "success_kinematic_z_contrast_by_finger_summary.csv")
ka.save_csv(subject_finger_trajectory, OUTPUT_ROOT, "subject_finger_trajectory.csv")
ka.save_csv(trajectory_variability_summary, OUTPUT_ROOT, "trajectory_variability_summary.csv")
ka.save_csv(finger_trajectory_distance_paired, OUTPUT_ROOT, "finger_trajectory_distance_paired.csv")
ka.save_csv(finger_trajectory_distance_summary, OUTPUT_ROOT, "finger_trajectory_distance_summary.csv")
ka.save_csv(success_failure_trajectory_distance, OUTPUT_ROOT, "success_failure_trajectory_distance.csv")
ka.save_csv(success_failure_trajectory_distance_summary, OUTPUT_ROOT, "success_failure_trajectory_distance_summary.csv")

advanced_figure_paths = ka.save_advanced_kinematic_figures(OUTPUT_ROOT, success_kinematic_z, trajectory_structure, fig_dpi=FIG_DPI)

print("Success-linked kinematic/Z contrasts")
display(success_kinematic_z_contrast_summary.sort_values("metric").head(80))
print("Between-finger trajectory distance summary")
display(finger_trajectory_distance_summary.head(80))
print("Correct-vs-incorrect trajectory distance summary")
display(success_failure_trajectory_distance_summary.head(80))
print(f"Saved {len(advanced_figure_paths)} advanced figures")


Success-linked kinematic/Z contrasts


,metric,n_paired_observations,mean_difference,median_difference,sem_difference,cohens_dz,sign_flip_p
0,duration_s,158,-5.166499e-01,-0.172689,1.230530e-01,-0.334022,0.000050
1,max_acceleration_cm_s2,158,-3.052464e+01,6.697492,1.794700e+01,-0.135310,0.088646
2,max_jerk_cm_s3,158,-1.018968e+03,117.203757,1.301125e+03,-0.062304,0.439178
3,max_r_workspace_cm,158,-4.173204e-01,-0.166840,1.001368e-01,-0.331549,0.000050
4,max_side_z_lift_cm,158,-3.557380e-01,-0.131185,1.435758e-01,-0.197115,0.013599
5,max_speed_cm_s,158,-1.407956e+00,-0.396507,6.629534e-01,-0.168957,0.032848
6,mean_abs_radial_velocity_cm_s,158,1.936271e-01,0.227854,1.104910e-01,0.139415,0.083696
7,mean_abs_tangential_velocity_cm_s,158,7.733594e-03,0.044758,4.075434e-02,0.015097,0.853307
8,mean_acceleration_cm_s2,158,7.150045e-01,0.894927,1.034924e+00,0.054963,0.495325
9,mean_curvature_1_cm,158,5.542465e+40,0.000502,5.542465e+40,0.079556,1.000000


Between-finger trajectory distance summary


,comparison,metric,n_subject_stiffness_pairs,mean,median,sem
0,M - I,mean_xy_trajectory_distance_cm,351,3.204410,2.955948,0.090120
1,M - I,rms_xy_trajectory_distance_cm,351,4.236618,3.891447,0.121219
2,M - I,speed_profile_rmse_cm_s,351,10.128245,8.935846,0.290450
3,P - I,mean_xy_trajectory_distance_cm,360,3.176032,2.966756,0.082692
4,P - I,rms_xy_trajectory_distance_cm,360,4.168073,3.842784,0.108292
5,P - I,speed_profile_rmse_cm_s,360,9.421160,8.170704,0.255986
6,P - M,mean_xy_trajectory_distance_cm,351,3.366638,3.203618,0.093743
7,P - M,rms_xy_trajectory_distance_cm,351,4.436071,4.203439,0.125875
8,P - M,speed_profile_rmse_cm_s,351,10.902756,9.340178,0.377225
9,P - R,mean_xy_trajectory_distance_cm,351,3.260893,2.958965,0.106761


Correct-vs-incorrect trajectory distance summary


,finger_condition,metric,n_subject_stiffness_pairs,mean,median,sem
0,I,success_failure_mean_xy_distance_cm,264,5.111712,4.587671,0.189958
1,I,success_failure_rms_xy_distance_cm,264,7.197120,6.553758,0.260246
2,I,success_failure_speed_rmse_cm_s,264,12.825429,11.045890,0.545857
3,M,success_failure_mean_xy_distance_cm,267,5.121091,4.608027,0.194185
4,M,success_failure_rms_xy_distance_cm,267,7.257035,6.449824,0.263964
5,M,success_failure_speed_rmse_cm_s,267,13.203804,11.208158,0.565543
6,P,success_failure_mean_xy_distance_cm,257,4.353796,3.954296,0.154785
7,P,success_failure_rms_xy_distance_cm,257,6.176734,5.719845,0.209691
8,P,success_failure_speed_rmse_cm_s,257,11.637728,10.827324,0.363344
9,R,success_failure_mean_xy_distance_cm,252,4.737704,4.113360,0.175454


Saved 3 advanced figures


## 7. Subject-specific XY trajectories in space

This is the subject-first view: one spatial XY trajectory table and one figure per subject. No pooling across participants is used in these outputs. Thin lines show that subject's stiffness-specific trajectories; thick lines show that same subject's finger-average trajectory.


In [12]:
subject_spatial = ka.compute_subject_spatial_trajectory_analysis(trajectory_time_bins)
subject_xy_trajectory = subject_spatial["subject_xy_trajectory"]
subject_spatial_trajectory_summary = subject_spatial["subject_spatial_trajectory_summary"]
subject_finger_spatial_distance = subject_spatial["subject_finger_spatial_distance"]
subject_spatial_metric_distribution = subject_spatial["subject_spatial_metric_distribution"]

ka.save_csv(subject_xy_trajectory, OUTPUT_ROOT, "subject_xy_trajectory.csv")
ka.save_csv(subject_spatial_trajectory_summary, OUTPUT_ROOT, "subject_spatial_trajectory_summary.csv")
ka.save_csv(subject_finger_spatial_distance, OUTPUT_ROOT, "subject_finger_spatial_distance.csv")
ka.save_csv(subject_spatial_metric_distribution, OUTPUT_ROOT, "subject_spatial_metric_distribution.csv")
subject_xy_figure_paths = ka.save_subject_xy_trajectory_figures(OUTPUT_ROOT, subject_xy_trajectory, fig_dpi=FIG_DPI)

print("Subject XY trajectory table:", subject_xy_trajectory.shape)
display(subject_xy_trajectory.head(30))
print("Subject-specific spatial trajectory summary:")
display(subject_spatial_trajectory_summary.sort_values(["subject_id", "finger_condition", "stiffness_value"]).head(80))
print("Subject-specific finger spatial distances:")
display(subject_finger_spatial_distance.sort_values(["subject_id", "stiffness_value", "comparison"]).head(80))
print("Subject spatial metric distributions with mean/median/95% CI and log-backtransformed summaries:")
display(subject_spatial_metric_distribution.head(80))
print(f"Saved {len(subject_xy_figure_paths)} per-subject XY figures")


Subject XY trajectory table: (71100, 25)


,subject_id,finger_condition,stiffness_value,trajectory_time_bin,n_segments,time_fraction,x_centered_px,y_centered_px,sd_x_centered_px,sd_y_centered_px,r_center_px,speed_px_s,subject_group,experiment_group,workspace_setup,workspace_label,workspace_width_cm,workspace_height_cm,x_workspace_cm,y_workspace_cm,r_workspace_cm,r_workspace_normalized,success_label,success_rate,xy_sd_radius_px
0,L_E_1,I,25.0,1,8,0.008130,0.000000,0.000000,0.000000,0.000000,0.000000,19.513735,L,L_E,L,L workspace (80x60 cm),80.0,60.0,0.000000,0.000000,0.000000,0.000000,success,1.0,0.000000
1,L_E_1,I,25.0,2,8,0.030007,0.000000,0.000000,0.000000,0.000000,0.000000,8.207517,L,L_E,L,L workspace (80x60 cm),80.0,60.0,0.000000,0.000000,0.000000,0.000000,success,1.0,0.000000
2,L_E_1,I,25.0,3,8,0.050044,0.000000,0.000000,0.000000,0.000000,0.000000,1.272294,L,L_E,L,L workspace (80x60 cm),80.0,60.0,0.000000,0.000000,0.000000,0.000000,success,1.0,0.000000
3,L_E_1,I,25.0,4,8,0.069877,0.000000,0.000000,0.000000,0.000000,0.000000,1.122887,L,L_E,L,L workspace (80x60 cm),80.0,60.0,0.000000,0.000000,0.000000,0.000000,success,1.0,0.000000
4,L_E_1,I,25.0,5,8,0.089672,0.000000,0.000000,0.000000,0.000000,0.000000,3.667713,L,L_E,L,L workspace (80x60 cm),80.0,60.0,0.000000,0.000000,0.000000,0.000000,success,1.0,0.000000
5,L_E_1,I,25.0,6,8,0.110164,0.000000,0.000000,0.000000,0.000000,0.000000,9.585927,L,L_E,L,L workspace (80x60 cm),80.0,60.0,0.000000,0.000000,0.000000,0.000000,success,1.0,0.000000
6,L_E_1,I,25.0,7,8,0.129873,0.223554,-0.152897,0.632305,0.432459,0.290784,15.835575,L,L_E,L,L workspace (80x60 cm),80.0,60.0,0.027944,-0.019112,0.033855,0.000677,success,1.0,0.766049
7,L_E_1,I,25.0,8,8,0.149511,-0.096285,-0.880466,2.705283,3.315101,2.423602,18.482263,L,L_E,L,L workspace (80x60 cm),80.0,60.0,-0.012036,-0.110058,0.303109,0.006062,success,1.0,4.278838
8,L_E_1,I,25.0,9,8,0.169710,-0.458347,-1.101515,3.051522,3.157734,2.501492,23.760156,L,L_E,L,L workspace (80x60 cm),80.0,60.0,-0.057293,-0.137689,0.313306,0.006266,success,1.0,4.391250
9,L_E_1,I,25.0,10,8,0.189807,0.854073,-2.484988,4.741200,3.199361,4.421308,35.598449,L,L_E,L,L workspace (80x60 cm),80.0,60.0,0.106759,-0.310624,0.547073,0.010941,success,1.0,5.719693


Subject-specific spatial trajectory summary:


,subject_id,finger_condition,stiffness_value,subject_group,experiment_group,workspace_setup,workspace_label,workspace_width_cm,workspace_height_cm,n_time_bins,n_segments_mean,start_x_centered_px,start_y_centered_px,end_x_centered_px,end_y_centered_px,centroid_x_centered_px,centroid_y_centered_px,min_x_centered_px,max_x_centered_px,min_y_centered_px,max_y_centered_px,xy_width_px,xy_height_px,spatial_extent_area_px2,mean_radius_px,max_radius_px,mean_trajectory_speed_px_s,mean_xy_sd_radius_px,mean_trajectory_path_length_px,mean_trajectory_net_displacement_px,mean_trajectory_straightness,mean_trajectory_signed_area_px2,success_rate
0,L_E_1,I,25.0,L,L_E,L,L workspace (80x60 cm),80.0,60.0,50,8.0,0.0,0.0,-4.368948,-3.473994,-16.859020,-45.454219,-135.770973,75.578109,-110.783865,0.000000,211.349082,110.783865,23414.068076,67.362018,152.720721,201.646488,46.084879,503.827508,5.581786,0.011079,-13537.651225,1.000000
1,L_E_1,I,40.0,L,L_E,L,L workspace (80x60 cm),80.0,60.0,50,8.0,0.0,0.0,4.988188,-4.596750,-1.359306,-49.355576,-58.776972,36.160442,-98.103607,0.000000,94.937414,98.103607,9313.702703,53.754414,106.551566,201.831596,76.367797,360.219429,6.783225,0.018831,-2756.034486,1.000000
2,L_E_1,I,55.0,L,L_E,L,L workspace (80x60 cm),80.0,60.0,50,8.0,0.0,0.0,1.052263,1.069826,11.907574,-44.873050,-57.392344,78.299987,-87.588254,1.069826,135.692332,88.658080,12030.221673,53.697252,113.798983,203.947888,77.702823,448.956423,1.500595,0.003342,-1306.015553,1.000000
3,L_E_1,I,70.0,L,L_E,L,L workspace (80x60 cm),80.0,60.0,50,8.0,0.0,0.0,0.359268,-5.020476,-6.859548,-52.850891,-77.773821,41.710284,-91.643793,0.616205,119.484105,92.259998,11023.603275,59.525775,114.960366,209.908286,71.313493,347.263473,5.033315,0.014494,-3229.259332,0.750000
4,L_E_1,I,85.0,L,L_E,L,L workspace (80x60 cm),80.0,60.0,50,64.0,0.0,0.0,-0.599369,-2.679370,-5.818512,-51.024158,-44.889940,50.418168,-85.663084,0.000000,95.308108,85.663084,8164.386427,55.948378,95.044197,215.617191,81.935420,316.223464,2.745591,0.008682,-2403.520536,0.890625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,L_E_11,I,70.0,L,L_E,L,L workspace (80x60 cm),80.0,60.0,50,8.0,0.0,0.0,1.862752,-4.160038,21.781008,-33.296724,0.000000,45.554199,-60.446818,0.000000,45.554199,60.446818,2753.606359,40.201201,72.228649,128.959307,63.786433,237.685808,4.558044,0.019177,-352.132122,0.500000
76,L_E_11,I,85.0,L,L_E,L,L workspace (80x60 cm),80.0,60.0,50,64.0,0.0,0.0,5.651117,-9.918803,18.984867,-34.217998,-3.472765,45.349555,-68.381341,0.000000,48.822320,68.381341,3338.535756,39.618442,81.169158,123.792320,60.977635,212.123005,11.415681,0.053816,745.570933,0.703125
77,L_E_11,I,100.0,L,L_E,L,L workspace (80x60 cm),80.0,60.0,50,8.0,0.0,0.0,7.132833,-12.073653,22.930529,-40.005424,-0.721418,61.779421,-86.238243,0.000000,62.500839,86.238243,5389.962566,46.520009,106.083606,102.884735,45.077465,324.031254,14.023210,0.043277,950.369330,0.625000
78,L_E_11,I,115.0,L,L_E,L,L workspace (80x60 cm),80.0,60.0,50,8.0,0.0,0.0,-3.418508,-4.476650,1.624180,-20.463913,-28.083685,34.924854,-39.463836,0.000000,63.008539,39.463836,2486.558641,24.813515,49.992183,126.720775,68.220300,370.334371,5.632636,0.015210,-186.599565,0.625000


Subject-specific finger spatial distances:


,subject_id,stiffness_value,finger_a,finger_b,comparison,n_matched_time_bins,mean_xy_distance_px,median_xy_distance_px,rms_xy_distance_px,max_xy_distance_px
0,L_E_1,25.0,I,M,M - I,50,24.917749,22.054145,32.494988,71.257294
2,L_E_1,25.0,I,P,P - I,50,40.155002,36.268803,52.837145,106.148972
4,L_E_1,25.0,M,P,P - M,50,40.740819,43.997292,49.776860,91.977325
5,L_E_1,25.0,R,P,P - R,50,81.596991,39.055656,120.153556,245.048975
1,L_E_1,25.0,I,R,R - I,50,111.010828,70.281965,160.413516,335.935059
...,...,...,...,...,...,...,...,...,...,...
77,L_E_10,70.0,R,P,P - R,50,28.078940,27.585585,35.387827,70.821789
73,L_E_10,70.0,I,R,R - I,50,37.411457,33.581305,45.313543,83.457640
75,L_E_10,70.0,M,R,R - M,50,37.908873,34.369629,47.458882,83.959128
78,L_E_10,85.0,I,M,M - I,50,24.343309,21.873163,29.749900,53.474904


Subject spatial metric distributions with mean/median/95% CI and log-backtransformed summaries:


,subject_id,finger_condition,metric,n,mean,mean_ci95_low,mean_ci95_high,median,median_ci95_low,median_ci95_high,sd,sem,skewness,log_transform_valid,log_transform_recommended,geometric_mean_backtransformed,geometric_ci95_low_backtransformed,geometric_ci95_high_backtransformed
0,L_E_1,I,mean_trajectory_path_length_px,9,437.864259,384.105230,491.623288,448.956423,347.263473,522.005390,82.284228,27.428076,-0.185071,True,False,430.684550,379.063123,489.335866
1,L_E_1,I,mean_trajectory_net_displacement_px,9,4.071413,2.957050,5.185777,3.845028,2.745591,5.581786,1.705658,0.568553,0.081610,True,False,3.719144,2.730564,5.065632
2,L_E_1,I,mean_trajectory_straightness,9,0.009776,0.006530,0.013022,0.008682,0.005664,0.014494,0.004968,0.001656,0.415525,True,False,0.008635,0.006049,0.012329
3,L_E_1,I,spatial_extent_area_px2,9,12638.036576,9047.518663,16228.554488,11023.603275,8164.386427,17906.866910,5495.690683,1831.896894,0.710766,True,False,11697.993733,8951.753472,15286.732125
4,L_E_1,I,mean_radius_px,9,58.133135,52.538159,63.728110,55.948378,52.179899,67.362018,8.563738,2.854579,0.036414,True,False,57.565642,52.213260,63.466697
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,L_E_11,P,max_radius_px,9,83.157095,68.818034,97.496155,86.668525,53.001647,102.141996,21.947542,7.315847,-0.313443,True,False,80.317861,66.557373,96.923279
76,L_E_11,P,mean_xy_sd_radius_px,9,59.092782,54.689821,63.495744,59.082595,52.100683,66.736478,6.739227,2.246409,0.202605,True,False,58.755142,54.563196,63.269144
77,L_E_11,R,mean_trajectory_path_length_px,9,323.652444,281.998462,365.306425,340.884828,249.512129,372.900322,63.756094,21.252031,-0.441972,True,False,317.460117,275.857472,365.336943
78,L_E_11,R,mean_trajectory_net_displacement_px,9,6.595363,5.645351,7.545375,7.016174,5.909651,7.656654,1.454100,0.484700,-1.248767,True,True,6.402733,5.330283,7.690960


Saved 42 per-subject XY figures


## 8. Subject-specific velocity and acceleration

Same subject-first structure as the XY spatial plots, now for velocity and acceleration. The profiles use derivatives computed inside each stiffness segment, so velocity/acceleration do not bridge object transitions.


In [13]:
subject_va = ka.compute_subject_velocity_acceleration_analysis(trajectory_time_bins, side_z_samples=side_z_samples, n_time_bins=TRAJECTORY_TIME_BINS)
subject_velocity_acceleration_profile = subject_va["subject_velocity_acceleration_profile"]
subject_velocity_acceleration_summary = subject_va["subject_velocity_acceleration_summary"]
subject_finger_velocity_acceleration_distance = subject_va["subject_finger_velocity_acceleration_distance"]
subject_velocity_acceleration_metric_distribution = subject_va["subject_velocity_acceleration_metric_distribution"]
velocity_stiffness_influence_summary = subject_va.get("velocity_stiffness_influence_summary", pd.DataFrame())
velocity_finger_influence_summary = subject_va.get("velocity_finger_influence_summary", pd.DataFrame())
velocity_time_influence_summary = subject_va.get("velocity_time_influence_summary", pd.DataFrame())

ka.save_csv(subject_velocity_acceleration_profile, OUTPUT_ROOT, "subject_velocity_acceleration_profile.csv")
ka.save_csv(subject_velocity_acceleration_summary, OUTPUT_ROOT, "subject_velocity_acceleration_summary.csv")
ka.save_csv(subject_finger_velocity_acceleration_distance, OUTPUT_ROOT, "subject_finger_velocity_acceleration_distance.csv")
ka.save_csv(subject_velocity_acceleration_metric_distribution, OUTPUT_ROOT, "subject_velocity_acceleration_metric_distribution.csv")
ka.save_csv(velocity_stiffness_influence_summary, OUTPUT_ROOT, "velocity_stiffness_influence_summary.csv")
ka.save_csv(velocity_finger_influence_summary, OUTPUT_ROOT, "velocity_finger_influence_summary.csv")
ka.save_csv(velocity_time_influence_summary, OUTPUT_ROOT, "velocity_time_influence_summary.csv")
subject_va_figure_paths = ka.save_subject_velocity_acceleration_figures(
    OUTPUT_ROOT,
    subject_velocity_acceleration_profile,
    fig_dpi=FIG_DPI,
)
standard_vs_comparison_figure_paths = ka.save_standard_vs_comparison_velocity_figures(
    OUTPUT_ROOT,
    subject_velocity_acceleration_profile,
    levels=("subject", "group"),
    fig_dpi=FIG_DPI,
)
print(f"Saved {len(standard_vs_comparison_figure_paths)} standard-vs-comparison velocity figures (per subject + per group, 8 metrics)")

print("Subject velocity/acceleration profile:", subject_velocity_acceleration_profile.shape)
display(subject_velocity_acceleration_profile.head(30))
print("Subject velocity/acceleration summary:")
display(subject_velocity_acceleration_summary.sort_values(["subject_id", "finger_condition", "stiffness_value"]).head(80))
print("Subject-specific finger velocity/acceleration distances:")
display(subject_finger_velocity_acceleration_distance.sort_values(["subject_id", "stiffness_value", "comparison"]).head(80))
print("Subject velocity/acceleration metric distributions with mean/median/95% CI and log-backtransformed summaries:")
display(subject_velocity_acceleration_metric_distribution.head(80))
print("Velocity influence answers: stiffness, fingers, and time thirds")
display(velocity_stiffness_influence_summary)
display(velocity_finger_influence_summary)
display(velocity_time_influence_summary.head(80))
print(f"Saved {len(subject_va_figure_paths)} velocity/acceleration figures for this selection level")


C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\kinematics_analysis.py:7780: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["tangential_radial_velocity_ratio_3d_proxy"] = np.where(
C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\kinematics_analysis.py:7780: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["tangential_radial_velocity_ratio_3d_proxy"] = np.where(


Saved 860 standard-vs-comparison velocity figures (per subject + per group, 8 metrics)
Subject velocity/acceleration profile: (71100, 131)


,subject_id,finger_condition,stiffness_value,trajectory_time_bin,n_segments,time_fraction,subject_group,experiment_group,workspace_setup,workspace_label,workspace_width_cm,workspace_height_cm,success_label,standard_value,comparison_value,success_rate,vx_cm_s,sd_vx_cm_s,vy_cm_s,sd_vy_cm_s,vz_cm_s,sd_vz_cm_s,vx_3d_cm_s,sd_vx_3d_cm_s,vy_3d_cm_s,sd_vy_3d_cm_s,vz_3d_proxy_cm_s,sd_vz_3d_proxy_cm_s,speed_cm_s,sd_speed_cm_s,speed_3d_proxy_cm_s,sd_speed_3d_proxy_cm_s,ax_cm_s2,sd_ax_cm_s2,ay_cm_s2,sd_ay_cm_s2,az_cm_s2,sd_az_cm_s2,ax_3d_cm_s2,sd_ax_3d_cm_s2,ay_3d_cm_s2,sd_ay_3d_cm_s2,az_3d_proxy_cm_s2,sd_az_3d_proxy_cm_s2,acceleration_cm_s2,sd_acceleration_cm_s2,acceleration_3d_proxy_cm_s2,sd_acceleration_3d_proxy_cm_s2,jx_cm_s3,sd_jx_cm_s3,jy_cm_s3,sd_jy_cm_s3,jerk_cm_s3,sd_jerk_cm_s3,radial_velocity_cm_s,sd_radial_velocity_cm_s,tangential_velocity_cm_s,sd_tangential_velocity_cm_s,x_workspace_cm,sd_x_workspace_cm,...,sd_vx_px_s,vy_px_s,sd_vy_px_s,vz_px_s,sd_vz_px_s,vx_3d_px_s,sd_vx_3d_px_s,vy_3d_px_s,sd_vy_3d_px_s,vz_3d_proxy_px_s,sd_vz_3d_proxy_px_s,speed_px_s,sd_speed_px_s,speed_3d_proxy_px_s,sd_speed_3d_proxy_px_s,ax_px_s2,sd_ax_px_s2,ay_px_s2,sd_ay_px_s2,az_px_s2,sd_az_px_s2,ax_3d_px_s2,sd_ax_3d_px_s2,ay_3d_px_s2,sd_ay_3d_px_s2,az_3d_proxy_px_s2,sd_az_3d_proxy_px_s2,acceleration_px_s2,sd_acceleration_px_s2,acceleration_3d_proxy_px_s2,sd_acceleration_3d_proxy_px_s2,jx_px_s3,sd_jx_px_s3,jy_px_s3,sd_jy_px_s3,jerk_px_s3,sd_jerk_px_s3,radial_velocity_px_s,sd_radial_velocity_px_s,tangential_velocity_px_s,sd_tangential_velocity_px_s,x_centered_px,sd_x_centered_px,y_centered_px,sd_y_centered_px,z_lift_px,sd_z_lift_px,abs_x_centered_px,sd_abs_x_centered_px,abs_y_centered_px,sd_abs_y_centered_px,abs_z_lift_px,sd_abs_z_lift_px,movement_angle_deg,velocity_heading_deg,velocity_heading_cm_deg,acceleration_heading_deg,acceleration_heading_cm_deg,speed_acceleration_product,speed_acceleration_product_cm
0,L_E_1,I,25.0,1,8,0.008130,L,L_E,L,L workspace (80x60 cm),80.0,60.0,success,85.0,25.0,1.0,0.319109,2.566959,0.774973,2.542645,NaN,NaN,0.319109,2.566959,0.774973,2.542645,NaN,NaN,2.436366,2.659553,NaN,NaN,-1.554260,26.405181,-15.789132,24.363923,NaN,NaN,-1.554260,26.405181,-15.789132,24.363923,NaN,NaN,24.873705,29.519799,NaN,NaN,-201.450963,750.128368,514.230368,1912.977570,912.652346,1902.324845,0.000000,0.000000,0.774973,2.542645,0.000000,0.000000,...,20.535672,6.199784,20.341158,NaN,NaN,2.552874,20.535672,6.199784,20.341158,NaN,NaN,19.513735,21.295806,NaN,NaN,-12.434078,211.241447,-126.313056,194.911382,NaN,NaN,-12.434078,211.241447,-126.313056,194.911382,NaN,NaN,230.323024,248.612357,NaN,NaN,-1611.607707,6001.026943,4113.842944,15303.820560,7565.380420,15147.271031,0.000000,0.000000,6.199784,20.341158,0.000000,0.000000,0.000000,0.000000,2.866305,0.0,0.000000,0.000000,0.000000,0.000000,2.866305,0.0,30.498464,67.619691,67.619691,-95.622002,-95.622002,4.494462e+03,60.601441
1,L_E_1,I,25.0,2,8,0.030007,L,L_E,L,L workspace (80x60 cm),80.0,60.0,success,85.0,25.0,1.0,0.204445,1.065734,0.123101,1.177919,1.519985,0.0,0.204445,1.065734,0.123101,1.177919,1.519985,0.0,1.025588,1.177403,2.045367,0.669027,-2.577875,25.280084,-7.120694,27.183076,NaN,NaN,-2.577875,25.280084,-7.120694,27.183076,NaN,NaN,25.055922,26.945501,NaN,NaN,-46.754065,181.281923,195.952029,337.739334,245.514736,352.736687,0.000000,0.000000,0.123101,1.177919,0.000000,0.000000,...,8.525871,0.984804,9.423350,12.159877,0.0,1.635557,8.525871,0.984804,9.423350,12.159877,0.0,8.207517,9.420805,16.362940,5.352216,-20.623000,202.240674,-56.965549,217.464610,NaN,NaN,-20.623000,202.240674,-56.965549,217.464610,NaN,NaN,200.849998,215.861459,NaN,NaN,-374.032520,1450.255386,1567.616229,2701.914672,2995.521843,3323.266482,0.000000,0.000000,0.984804,9.423350,0.000000,0.000000,0.000000,0.000000,3.196594,0.0,0.000000,0.000000,0.000000,0.000000,3.196594,0.0,49.524744,31.053047,31.053047,-109.901565,-109.901565,1.648480e+03,25.697041
2,L_E_1,I,25.0,3,8,0.050044,L,L_E,L,L workspace (80x60 cm),80.0,60.0,success,8

Subject velocity/acceleration summary:


,subject_id,finger_condition,stiffness_value,subject_group,experiment_group,workspace_setup,workspace_label,n_time_bins,n_segments_mean,success_rate,mean_vx_cm_s,peak_abs_vx_cm_s,sd_over_time_vx_cm_s,time_fraction_peak_abs_vx_cm_s,mean_vy_cm_s,peak_abs_vy_cm_s,sd_over_time_vy_cm_s,time_fraction_peak_abs_vy_cm_s,mean_vz_cm_s,peak_abs_vz_cm_s,sd_over_time_vz_cm_s,time_fraction_peak_abs_vz_cm_s,mean_vx_3d_cm_s,peak_abs_vx_3d_cm_s,sd_over_time_vx_3d_cm_s,time_fraction_peak_abs_vx_3d_cm_s,mean_vy_3d_cm_s,peak_abs_vy_3d_cm_s,sd_over_time_vy_3d_cm_s,time_fraction_peak_abs_vy_3d_cm_s,mean_vz_3d_proxy_cm_s,peak_abs_vz_3d_proxy_cm_s,sd_over_time_vz_3d_proxy_cm_s,time_fraction_peak_abs_vz_3d_proxy_cm_s,mean_speed_3d_proxy_cm_s,peak_abs_speed_3d_proxy_cm_s,sd_over_time_speed_3d_proxy_cm_s,time_fraction_peak_abs_speed_3d_proxy_cm_s,mean_speed_cm_s,peak_abs_speed_cm_s,sd_over_time_speed_cm_s,time_fraction_peak_abs_speed_cm_s,mean_ax_cm_s2,peak_abs_ax_cm_s2,sd_over_time_ax_cm_s2,time_fraction_peak_abs_ax_cm_s2,mean_ay_cm_s2,peak_abs_ay_cm_s2,sd_over_time_ay_cm_s2,time_fraction_peak_abs_ay_cm_s2,mean_az_cm_s2,peak_abs_az_cm_s2,sd_over_time_az_cm_s2,time_fraction_peak_abs_az_cm_s2,mean_ax_3d_cm_s2,peak_abs_ax_3d_cm_s2,sd_over_time_ax_3d_cm_s2,time_fraction_peak_abs_ax_3d_cm_s2,mean_ay_3d_cm_s2,peak_abs_ay_3d_cm_s2,...,mean_ay_px_s2,peak_abs_ay_px_s2,sd_over_time_ay_px_s2,time_fraction_peak_abs_ay_px_s2,mean_az_px_s2,peak_abs_az_px_s2,sd_over_time_az_px_s2,time_fraction_peak_abs_az_px_s2,mean_ax_3d_px_s2,peak_abs_ax_3d_px_s2,sd_over_time_ax_3d_px_s2,time_fraction_peak_abs_ax_3d_px_s2,mean_ay_3d_px_s2,peak_abs_ay_3d_px_s2,sd_over_time_ay_3d_px_s2,time_fraction_peak_abs_ay_3d_px_s2,mean_az_3d_proxy_px_s2,peak_abs_az_3d_proxy_px_s2,sd_over_time_az_3d_proxy_px_s2,time_fraction_peak_abs_az_3d_proxy_px_s2,mean_acceleration_3d_proxy_px_s2,peak_abs_acceleration_3d_proxy_px_s2,sd_over_time_acceleration_3d_proxy_px_s2,time_fraction_peak_abs_acceleration_3d_proxy_px_s2,mean_acceleration_px_s2,peak_abs_acceleration_px_s2,sd_over_time_acceleration_px_s2,time_fraction_peak_abs_acceleration_px_s2,mean_jx_px_s3,peak_abs_jx_px_s3,sd_over_time_jx_px_s3,time_fraction_peak_abs_jx_px_s3,mean_jy_px_s3,peak_abs_jy_px_s3,sd_over_time_jy_px_s3,time_fraction_peak_abs_jy_px_s3,mean_jerk_px_s3,peak_abs_jerk_px_s3,sd_over_time_jerk_px_s3,time_fraction_peak_abs_jerk_px_s3,mean_radial_velocity_px_s,peak_abs_radial_velocity_px_s,sd_over_time_radial_velocity_px_s,time_fraction_peak_abs_radial_velocity_px_s,mean_tangential_velocity_px_s,peak_abs_tangential_velocity_px_s,sd_over_time_tangential_velocity_px_s,time_fraction_peak_abs_tangential_velocity_px_s,early_mean_speed_cm_s,late_mean_speed_cm_s,late_minus_early_speed_cm_s,early_mean_acceleration_cm_s2,late_mean_acceleration_cm_s2,late_minus_early_acceleration_cm_s2,early_mean_speed_px_s,late_mean_speed_px_s,late_minus_early_speed_px_s,early_mean_acceleration_px_s2,late_mean_acceleration_px_s2,late_minus_early_acceleration_px_s2
0,L_E_1,I,25.0,L,L_E,L,L workspace (80x60 cm),50,8.0,1.000000,0.003769,53.019371,23.087357,0.570305,-0.023934,22.129792,11.057053,0.350331,0.587074,7.831731,2.778299,0.529736,0.003769,53.019371,23.087357,0.570305,-0.023934,22.129792,11.057053,0.350331,0.587074,7.831731,2.778299,0.529736,24.725717,54.334480,15.709965,0.590626,25.171138,55.589523,16.293163,0.570305,3.924342,199.802007,91.338208,0.469997,0.138126,75.495187,33.972353,0.269103,-0.019574,34.097262,12.476669,0.509759,3.924342,199.802007,91.338208,0.469997,0.138126,75.495187,...,1.105004,603.961500,271.778823,0.269103,-0.156591,272.778098,99.813352,0.509759,31.394733,1598.416057,730.705666,0.469997,1.105004,603.961500,271.778823,0.269103,-0.156591,272.778098,99.813352,0.509759,1047.341339,2004.971993,597.087281,0.529736,1372.882500,3238.605386,856.136710,0.549942,519.447756,78623.853340,22983.383413,0.570305,-1388.133921,57729.385921,14564.206724,0.390078,125653.216033,380809.311555,92948.207656,0.549942,-1.727903,300.839204,145.821868,0.82998

...

Velocity influence answers: stiffness, fingers, and time thirds


,stiffness_value,n_observations,n_subjects,median_vx_cm_s,median_vy_cm_s,median_vz_3d_proxy_cm_s,mean_velocity_cm_s,mean_velocity_3d_proxy_cm_s,sem_velocity_cm_s,median_ax_cm_s2,median_ay_cm_s2,median_az_3d_proxy_cm_s2,mean_acceleration_cm_s2,mean_acceleration_3d_proxy_cm_s2
0,25.0,7900,40,0.026569,0.017860,0.039324,16.026067,17.403293,0.172774,0.182984,0.000004,-0.778278,80.148022,94.478430
1,40.0,7900,40,0.039628,0.005293,0.043133,16.375818,17.557667,0.177060,0.031329,-0.108705,-0.811479,79.584548,91.278123
2,55.0,7900,40,0.038005,-0.002806,0.022193,16.315893,17.501709,0.175811,0.394813,-0.414177,-0.599329,78.684514,91.611683
3,70.0,7900,40,0.019845,0.000025,0.024912,16.151571,17.616253,0.178529,0.199687,-0.059497,-0.703443,77.525765,91.356999
4,85.0,7900,40,0.024020,0.007421,0.093965,16.435578,17.495118,0.166902,0.305895,-0.054424,-0.432879,81.106439,90.493467
5,100.0,7900,40,0.019685,0.005197,0.020475,16.229138,17.463988,0.174378,0.439774,0.000044,-0.566100,78.220144,91.094625
6,115.0,7900,40,0.027233,-0.018096,0.000000,15.699249,16.807764,0.167989,0.034057,0.262144,-0.790261,74.989543,88.834539
7,130.0,7900,40,0.023298,0.006847,0.069575,15.746290,17.163601,0.175701,0.263097,-0.117474,-0.634827,76.929391,90.797747
8,145.0,7900,40,0.015909,0.003378,0.037927,16.005650,17.367505,0.176309,0.153983,-0.169493,-0.648842,76.363793,89.884019


,finger_condition,n_observations,n_subjects,median_vx_cm_s,median_vy_cm_s,median_vz_3d_proxy_cm_s,mean_velocity_cm_s,mean_velocity_3d_proxy_cm_s,sem_velocity_cm_s,median_ax_cm_s2,median_ay_cm_s2,median_az_3d_proxy_cm_s2,mean_acceleration_cm_s2,mean_acceleration_3d_proxy_cm_s2
0,I,18000,40,0.038672,0.000254,0.059767,16.338686,17.716935,0.113456,0.201773,-0.159302,-0.677214,80.425130,94.286136
1,M,17550,39,0.009168,0.003046,0.039302,15.904082,17.283195,0.130956,0.042684,-0.031855,-0.655520,79.321207,94.446198
2,P,18000,40,0.042671,0.003242,0.018609,15.844406,16.919426,0.095815,0.571506,-0.033367,-0.537729,74.445843,84.716761
3,R,17550,39,0.017803,0.008057,0.038463,16.384685,17.576569,0.121129,0.082873,-0.000052,-0.783823,79.694947,92.147786


,time_third,stiffness_value,n_observations,n_subjects,median_vx_cm_s,median_vy_cm_s,median_vz_3d_proxy_cm_s,mean_velocity_cm_s,mean_velocity_3d_proxy_cm_s,mean_acceleration_cm_s2,mean_acceleration_3d_proxy_cm_s2
0,early,25.0,2686,40,0.031654,-0.024227,0.150105,1.954403,3.952501,22.095588,36.708622
1,early,40.0,2686,40,0.043289,-0.035605,0.181681,2.181495,4.281686,24.125978,38.826160
2,early,55.0,2686,40,0.041167,-0.024407,0.104176,2.394333,4.399116,25.544564,41.626380
3,early,70.0,2686,40,0.014781,-0.016933,0.120956,2.607412,4.895895,26.003566,41.545891
4,early,85.0,2686,40,0.039628,-0.015465,0.207339,2.951165,4.784750,27.287619,42.285201
5,early,100.0,2686,40,0.043544,-0.013984,0.076579,2.534996,4.734004,25.950523,42.174756
6,early,115.0,2686,40,0.045126,-0.052789,0.017261,2.590433,4.420893,27.182569,43.484037
7,early,130.0,2686,40,0.028128,-0.004333,0.219855,2.385461,4.768608,25.507408,40.139518
8,early,145.0,2686,40,0.029553,-0.033382,0.176856,2.587031,4.714003,26.021745,41.013286
9,middle,25.0,2528,40,1.062896,0.335308,0.031679,21.613926,22.618189,108.716558,116.544192


Saved 525 velocity/acceleration figures for this selection level


In [14]:
standard_vs_comparison_position_figure_paths = ka.save_standard_vs_comparison_position_figures(
    OUTPUT_ROOT, subject_velocity_acceleration_profile, levels=("subject", "group"), fig_dpi=FIG_DPI
)
print(
    f"Saved {len(standard_vs_comparison_position_figure_paths)} standard-vs-comparison position figures "
    "(|Y| and |X| distance from center + movement direction; per subject + per group)"
)


Saved 215 standard-vs-comparison position figures (|Y| and |X| distance from center + movement direction; per subject + per group)


## 9. 3D proxy kinematics, confidence intervals, and optional LMM

Combines top-camera XY with side-camera Z/lift proxy inside each subject/trial/stiffness segment. Outputs include 3D path length, max excursion, peak velocity, peak acceleration, median and 95% confidence intervals, log-domain summaries back-transformed to original units, individual data points in the direction/stiffness figure, and an optional mixed-effects model if `statsmodels` is installed.


In [15]:
proxy3d = ka.compute_3d_proxy_kinematics(kinematic_samples, side_z_samples)
kinematic_3d_proxy_samples = proxy3d["kinematic_3d_proxy_samples"]
trial_3d_kinematic_summary = proxy3d["trial_3d_kinematic_summary"]
subject_3d_kinematic_summary = proxy3d["subject_3d_kinematic_summary"]
subject_3d_metric_distribution = proxy3d["subject_3d_metric_distribution"]
stiffness_direction_3d_metric_distribution = proxy3d["stiffness_direction_3d_metric_distribution"]
mixedlm_model_input = proxy3d["mixedlm_model_input"]

ka.save_parquet(kinematic_3d_proxy_samples, OUTPUT_ROOT, "kinematic_3d_proxy_samples.parquet")
ka.save_csv(trial_3d_kinematic_summary, OUTPUT_ROOT, "trial_3d_kinematic_summary.csv")
ka.save_csv(subject_3d_kinematic_summary, OUTPUT_ROOT, "subject_3d_kinematic_summary.csv")
ka.save_csv(subject_3d_metric_distribution, OUTPUT_ROOT, "subject_3d_metric_distribution.csv")
ka.save_csv(stiffness_direction_3d_metric_distribution, OUTPUT_ROOT, "stiffness_direction_3d_metric_distribution.csv")
ka.save_csv(mixedlm_model_input, OUTPUT_ROOT, "mixedlm_model_input.csv")

mixedlm = ka.fit_optional_mixed_effects_models(mixedlm_model_input)
ka.save_csv(mixedlm["mixedlm_status"], OUTPUT_ROOT, "mixedlm_status.csv")
ka.save_csv(mixedlm["mixedlm_coefficients"], OUTPUT_ROOT, "mixedlm_coefficients.csv")
proxy_3d_figure_paths = ka.save_3d_proxy_figures(OUTPUT_ROOT, kinematic_3d_proxy_samples, trial_3d_kinematic_summary, fig_dpi=FIG_DPI)

print("3D proxy trial summary:", trial_3d_kinematic_summary.shape)
display(trial_3d_kinematic_summary.head(30))
print("3D subject summaries:")
display(subject_3d_kinematic_summary.head(30))
print("3D metric distributions: mean, median, 95% CI, log-transform recommendation, and back-transformed geometric mean/CI")
display(subject_3d_metric_distribution.head(80))
print("MixedLM status / coefficients")
display(mixedlm["mixedlm_status"])
display(mixedlm["mixedlm_coefficients"].head(80))
print(f"Saved {len(proxy_3d_figure_paths)} 3D proxy figures")


c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2245: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWar

3D proxy trial summary: (20478, 38)


,subject_id,subject_group,trial_index_raw,pair_number,finger_condition,stiffness_value,stiffness_segment_id,correct_response,dominant_movement_direction,n_3d_samples,path_length_3d_proxy_px,path_length_3d_proxy_cm,max_excursion_3d_from_start_px,max_excursion_3d_from_start_cm,max_radius_3d_from_center_px,max_radius_3d_from_center_cm,peak_velocity_3d_proxy_px_s,peak_velocity_3d_proxy_cm_s,mean_velocity_3d_proxy_px_s,mean_velocity_3d_proxy_cm_s,peak_acceleration_3d_proxy_px_s2,peak_acceleration_3d_proxy_cm_s2,mean_acceleration_3d_proxy_px_s2,mean_acceleration_3d_proxy_cm_s2,mean_z_lift_px,mean_z_lift_cm,max_z_lift_px,max_z_lift_cm,mean_side_lateral_camera_corrected_px,mean_side_lateral_camera_corrected_cm,path_length_side_view_camera_corrected_px,path_length_side_view_camera_corrected_cm,mean_side_lift_lateral_angle_camera_corrected_deg,mean_hand_orientation_xy_deg,mean_hand_orientation_yz_deg,mean_hand_orientation_zx_deg,straightness_3d_proxy,straightness_3d_proxy_cm
0,L_E_1,L,13,13,M,115.0,1,1.0,W,395,526.908247,65.863531,160.872087,20.109011,162.883645,20.360456,274.172969,34.271621,85.326562,10.665820,3993.987751,499.248469,558.220837,69.777605,11.529791,1.441224,29.233342,3.654168,NaN,NaN,0.0,0.0,NaN,-58.124665,171.291069,75.198887,0.305313,0.305313
1,L_E_1,L,13,13,M,85.0,2,1.0,ESE,311,590.221979,73.777747,153.576432,19.197054,155.699888,19.462486,278.861226,34.857653,120.865814,15.108227,4441.325059,555.165632,914.982277,114.372785,18.861240,2.357655,29.973501,3.746688,NaN,NaN,0.0,0.0,NaN,-56.502679,165.954122,68.508949,0.260201,0.260201
2,L_E_1,L,14,14,M,55.0,1,1.0,SSW,395,705.721291,88.215161,223.353598,27.919200,225.205565,28.150696,310.862869,38.857859,115.337282,14.417160,5123.882625,640.485328,735.087889,91.885986,16.440788,2.055099,30.198101,3.774763,NaN,NaN,0.0,0.0,NaN,-63.057167,168.526889,65.722761,0.316490,0.316490
3,L_E_1,L,14,14,M,85.0,2,1.0,S,250,605.198276,75.649785,165.652836,20.706605,168.095300,21.011913,427.251459,53.406432,155.622023,19.452753,4923.110570,615.388821,1032.535875,129.066984,12.689004,1.586126,26.989447,3.373681,NaN,NaN,0.0,0.0,NaN,-58.716775,170.662117,72.958058,0.273717,0.273717
4,L_E_1,L,15,15,M,40.0,1,1.0,S,287,567.374557,70.921820,168.741021,21.092628,166.347454,20.793432,309.818514,38.727314,126.195894,15.774487,6022.254961,752.781870,904.230381,113.028798,20.052793,2.506599,37.962569,4.745321,NaN,NaN,0.0,0.0,NaN,-54.997717,165.732080,66.941464,0.297407,0.297407
5,L_E_1,L,15,15,M,85.0,2,1.0,ESE,190,514.261438,64.282680,157.267107,19.658388,161.270554,20.158819,405.830351,50.728794,175.060946,21.882618,5172.632670,646.579084,1234.775489,154.346936,25.421553,3.177694,37.718462,4.714808,NaN,NaN,0.0,0.0,NaN,-59.706511,161.563747,59.587359,0.305812,0.305812
6,L_E_1,L,16,16,M,145.0,1,1.0,ESE,290,524.700751,65.587594,152.457515,19.057189,150.193134,18.774142,361.944289,45.243036,116.178122,14.522265,3830.581917,478.822740,782.801735,97.850217,10.305192,1.288149,22.395633,2.799454,NaN,NaN,0.0,0.0,NaN,-56.368057,172.297393,76.259870,0.290561,0.290561
7,L_E_1,L,16,16,M,85.0,2,1.0,SE,228,577.444868,72.180609,184.922329,23.115291,181.149236,22.643655,358.842445,44.855306,163.230306,20.403788,4867.744153,608.468019,1061.508630,132.688579,14.904415,1.863052,27.700078,3.462510,NaN,NaN,0.0,0.0,NaN,-60.188052,169.208768,69.995515,0.320242,0.320242
8,L_E_1,L,17,17,M,85.0,1,1.0,S,325,715.837814,89.479727,201.981620,25.247702,205.027507,25.628438,409.944222,51.243028,140.147680,17.518460,5915.496639,739.437080,980.290673,122.536334,20.166107,2.520763,35.345803,4.418225,NaN,NaN,0.0,0.0,NaN,-62.980198,165.879654,61.861207,0.282161,0.282161
9,L_E_1,L,17,17,M,130.0,2,1.0,S,270,739.158380,92.394798,220.334758,27.541845,223.990083,27.998760,515.019302,64.377413,174.877911,21.859739,8779.723138,1097.465392,1277.495957,159.686995,18.983478,2.372935,35.311958,4.413995,NaN,NaN,0.0,0.0,NaN,-63.196114,166.405301,62.028746,0.298089,0.298089


3D subject summaries:


,subject_id,subject_group,finger_condition,stiffness_value,n_trials,success_rate,path_length_3d_proxy_px,path_length_3d_proxy_cm,max_excursion_3d_from_start_px,max_excursion_3d_from_start_cm,max_radius_3d_from_center_px,max_radius_3d_from_center_cm,peak_velocity_3d_proxy_px_s,peak_velocity_3d_proxy_cm_s,mean_velocity_3d_proxy_px_s,mean_velocity_3d_proxy_cm_s,peak_acceleration_3d_proxy_px_s2,peak_acceleration_3d_proxy_cm_s2,mean_z_lift_px,mean_z_lift_cm,max_z_lift_px,max_z_lift_cm,mean_side_lateral_camera_corrected_px,mean_side_lateral_camera_corrected_cm,path_length_side_view_camera_corrected_px,path_length_side_view_camera_corrected_cm,mean_side_lift_lateral_angle_camera_corrected_deg,mean_hand_orientation_xy_deg,mean_hand_orientation_yz_deg,mean_hand_orientation_zx_deg,straightness_3d_proxy,straightness_3d_proxy_cm
0,L_E_1,L,I,25.0,8,1.000000,637.496514,79.687064,189.534404,23.691800,188.532564,23.566570,572.678519,71.584815,193.952758,24.244095,8503.141275,1062.892659,13.418194,1.677274,30.530685,3.816336,NaN,NaN,0.0,0.0,NaN,-62.584462,165.827826,62.678055,0.301844,0.301844
1,L_E_1,L,I,40.0,8,1.000000,828.865064,103.608133,197.002615,24.625327,196.027459,24.503432,570.460480,71.307560,204.755278,25.594410,9561.193318,1195.149165,11.088442,1.386055,28.507103,3.563388,NaN,NaN,0.0,0.0,NaN,-61.735252,166.913269,63.643450,0.258412,0.258412
2,L_E_1,L,I,55.0,8,1.000000,781.287939,97.660992,205.312566,25.664071,205.562902,25.695363,541.143007,67.642876,212.587512,26.573439,7990.942778,998.867847,24.107417,3.013427,36.942443,4.617805,NaN,NaN,0.0,0.0,NaN,-62.851125,159.986702,53.337413,0.283532,0.283532
3,L_E_1,L,I,70.0,8,0.750000,821.453786,102.681723,180.488688,22.561086,180.471875,22.558984,586.907571,73.363446,201.777610,25.222201,8596.894837,1074.611855,10.147115,1.268389,29.041781,3.630223,NaN,NaN,0.0,0.0,NaN,-61.870606,166.152436,62.495529,0.233977,0.233977
4,L_E_1,L,I,85.0,64,0.890625,1023.519582,127.939948,200.098942,25.012368,199.829732,24.978717,605.527653,75.690957,217.451965,27.181496,9869.603718,1233.700465,14.085759,1.760720,33.891491,4.236436,NaN,NaN,0.0,0.0,NaN,-63.647730,162.328363,56.050091,0.211247,0.211247
5,L_E_1,L,I,100.0,8,0.750000,1107.226084,138.403260,191.204596,23.900574,190.967770,23.870971,617.879961,77.234995,213.093894,26.636737,8841.174414,1105.146802,17.825269,2.228159,32.486711,4.060839,NaN,NaN,0.0,0.0,NaN,-63.326168,163.804670,57.423426,0.180174,0.180174
6,L_E_1,L,I,115.0,8,0.750000,1266.261497,158.282687,200.886404,25.110800,199.315029,24.914379,561.063510,70.132939,221.032391,27.629049,9438.763805,1179.845476,14.735436,1.841930,31.654581,3.956823,NaN,NaN,0.0,0.0,NaN,-63.713370,164.143070,58.336113,0.176379,0.176379
7,L_E_1,L,I,130.0,8,0.875000,1069.887428,133.735928,221.976367,27.747046,220.858103,27.607263,640.593886,80.074236,223.240646,27.905081,10680.621509,1335.077689,23.849033,2.981129,49.130982,6.141373,NaN,NaN,0.0,0.0,NaN,-61.523286,153.151037,48.308062,0.218486,0.218486
8,L_E_1,L,I,145.0,8,1.000000,947.678483,118.459810,198.974121,24.871765,197.926242,24.740780,607.571894,75.946487,211.748253,26.468532,10483.005943,1310.375743,15.507771,1.938471,30.488371,3.811046,NaN,NaN,0.0,0.0,NaN,-63.534267,165.258936,60.527841,0.217032,0.217032
9,L_E_1,L,M,25.0,8,1.000000,795.891320,99.486415,210.605542,26.325693,210.271935,26.283992,526.229704,65.778713,189.903647,23.737956,7274.386877,909.298360,22.922026,2.865253,46.766400,5.845800,NaN,NaN,0.0,0.0,NaN,-57.282724,160.144254,58.976914,0.283282,0.283282


...

,subject_id,finger_condition,metric,n,mean,mean_ci95_low,mean_ci95_high,median,median_ci95_low,median_ci95_high,sd,sem,skewness,log_transform_valid,log_transform_recommended,geometric_mean_backtransformed,geometric_ci95_low_backtransformed,geometric_ci95_high_backtransformed
0,L_E_1,I,path_length_3d_proxy_cm,9,117.828839,101.978453,133.679225,118.459810,97.660992,138.403260,24.260795,8.086932,0.077258,True,False,115.563938,100.682424,132.645036
1,L_E_1,I,max_excursion_3d_from_start_cm,9,24.798315,23.855531,25.741099,24.871765,23.691800,25.664071,1.443037,0.481012,0.482455,True,False,24.761628,23.848630,25.709577
2,L_E_1,I,max_radius_3d_from_center_cm,9,24.715162,23.786581,25.643743,24.740780,23.566570,25.695363,1.421298,0.473766,0.503382,True,False,24.679484,23.780647,25.612293
3,L_E_1,I,peak_velocity_3d_proxy_cm_s,9,73.664257,71.114478,76.214035,73.363446,70.132939,77.234995,3.902722,1.300907,0.083459,True,False,73.572487,71.070547,76.162504
4,L_E_1,I,peak_acceleration_3d_proxy_cm_s2,9,1166.185300,1091.096286,1241.274314,1179.845476,1062.892659,1310.375743,114.932165,38.310722,0.100162,True,False,1161.159939,1088.714068,1238.426547
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,L_E_10,M,mean_z_lift_cm,9,1.421268,1.167056,1.675480,1.264873,1.181537,1.616005,0.389100,0.129700,1.508725,True,True,1.383730,1.188576,1.610926
76,L_E_10,M,max_z_lift_cm,9,2.236509,2.027224,2.445793,2.298877,1.914528,2.448796,0.320334,0.106778,0.342152,True,False,2.216602,2.021264,2.430818
77,L_E_10,M,mean_side_lateral_camera_corrected_cm,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN,NaN
78,L_E_10,M,path_length_side_view_camera_corrected_cm,9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,NaN,NaN,NaN


MixedLM status / coefficients


,metric,status,reason,n_rows,n_subjects,n_directions,aic,bic
0,max_excursion_3d_from_start_px,fit,,20478,40,12,211546.902331,211753.007098
1,max_radius_3d_from_center_px,fit,,20478,40,12,212363.753016,212569.857782
2,path_length_3d_proxy_px,failed,Singular matrix,20478,40,12,NaN,NaN
3,peak_velocity_3d_proxy_px_s,failed,Singular matrix,20478,40,12,NaN,NaN


,metric,term,estimate,std_error,p_value
0,max_excursion_3d_from_start_px,Intercept,196.044759,8.606015,7.243958e-115
1,max_excursion_3d_from_start_px,C(direction_factor)[T.ENE],1.340666,5.026146,7.896706e-01
2,max_excursion_3d_from_start_px,C(direction_factor)[T.ESE],6.888170,6.403221,2.820457e-01
3,max_excursion_3d_from_start_px,C(direction_factor)[T.NE],7.404939,4.776258,1.210542e-01
4,max_excursion_3d_from_start_px,C(direction_factor)[T.NNE],16.458686,4.267046,1.147114e-04
5,max_excursion_3d_from_start_px,C(direction_factor)[T.S],10.312071,4.790791,3.135974e-02
6,max_excursion_3d_from_start_px,C(direction_factor)[T.SE],9.006813,4.595515,5.000587e-02
7,max_excursion_3d_from_start_px,C(direction_factor)[T.SSE],15.298733,4.357534,4.466440e-04
8,max_excursion_3d_from_start_px,C(direction_factor)[T.SSW],-3.228614,5.118332,5.281751e-01
9,max_excursion_3d_from_start_px,C(direction_factor)[T.SW],-0.579516,5.580924,9.172974e-01


Saved 41 3D proxy figures


## 10. Hand / active-finger orientation planes

Summarizes the thumb-to-active-finger orientation in XY from the top camera and YZ/ZX from the side-camera Z/lift proxy. The figures include three side-by-side vector panels (XY, YZ, ZX): faded vectors are individual trials and thick colored vectors are group means, matching common motor-control vector-field / mean-vector visualization practice. Raw pixel/proxy columns are preserved for traceability, while workspace labels document the setup calibration (`L` Lab/airslide = 80x60 cm; `N` Natural = 60x45 cm; top camera 100 cm above the table; side camera 10 cm above the table).


In [16]:
hand_orientation = ka.compute_hand_orientation_plane_analysis(trial_kinematic_summary, side_z_trial_summary)
hand_orientation_plane_trials = hand_orientation["hand_orientation_plane_trials"]
hand_orientation_plane_summary = hand_orientation["hand_orientation_plane_summary"]

ka.save_parquet(hand_orientation_plane_trials, OUTPUT_ROOT, "hand_orientation_plane_trials.parquet")
ka.save_csv(hand_orientation_plane_summary, OUTPUT_ROOT, "hand_orientation_plane_summary.csv")
hand_orientation_figure_paths = ka.save_hand_orientation_plane_figures(
    OUTPUT_ROOT, hand_orientation_plane_summary, hand_orientation_plane_trials, fig_dpi=FIG_DPI
)
hand_orientation_axis_matrix_paths = ka.save_hand_orientation_axis_matrix_figures(
    OUTPUT_ROOT, hand_orientation_plane_trials, fig_dpi=FIG_DPI
)

print("Hand orientation plane trials:", hand_orientation_plane_trials.shape)
display(hand_orientation_plane_summary.head(30))
print(f"Saved {len(hand_orientation_figure_paths)} hand-orientation figures")
print(f"Saved {len(hand_orientation_axis_matrix_paths)} hand-orientation axis matrix figures")


Hand orientation plane trials: (20480, 126)


,scope,group,plane,metric,n,circular_mean_deg,median_deg,resultant_length
0,all,all,XY,hand_orientation_xy_deg,20480,-51.784810,-54.378976,0.951733
1,all,all,YZ,hand_orientation_yz_deg,20478,148.107225,150.743180,0.932247
2,all,all,ZX,hand_orientation_zx_deg,20478,52.466145,54.935782,0.921356
3,experiment_group,L_E,XY,hand_orientation_xy_deg,10240,-49.181909,-52.529848,0.939927
4,experiment_group,L_E,YZ,hand_orientation_yz_deg,10240,145.391921,148.612939,0.908223
5,experiment_group,L_E,ZX,hand_orientation_zx_deg,10240,52.653272,55.062817,0.911331
6,experiment_group,N_E,XY,hand_orientation_xy_deg,10240,-54.318847,-55.741010,0.965452
7,experiment_group,N_E,YZ,hand_orientation_yz_deg,10238,150.681141,152.287438,0.958263
8,experiment_group,N_E,ZX,hand_orientation_zx_deg,10238,52.283013,54.799379,0.931392
9,subject_group,L,XY,hand_orientation_xy_deg,10240,-49.181909,-52.529848,0.939927


Saved 43 hand-orientation figures
Saved 43 hand-orientation axis matrix figures


## 11. Figures

All figures are saved under `analysis/Kinematics/result_fillter/figures`, including polar radiation patterns per participant and for all/E/P, side-by-side success/failure polar dwell-time plots with semi-transparent stiffness colors, and side-Z participant plots over time and over stiffness. Group summary graphs include faded raw values behind the summary bars when raw values are available.


In [17]:
group_output_side_z_samples = None if ka.is_single_subject_selection(DATA_SELECTION, _selected_subjects) else side_z_samples
movement_cycle_angle_figure_paths = ka.save_movement_cycle_hand_angle_figures(
    OUTPUT_ROOT,
    trajectory_time_bins,
    side_z_samples=group_output_side_z_samples,
    fig_dpi=FIG_DPI,
)
print(f"Saved {len(movement_cycle_angle_figure_paths)} movement-cycle hand-angle figures")

figure_paths = ka.save_kinematic_figures(
    OUTPUT_ROOT,
    trial_kinematic_summary,
    group_time_summary,
    direction_success_summary,
    distance_success_summary,
    side_z_group_time_summary,
    side_z_by_stiffness_summary,
    side_z_samples=side_z_samples,
    fig_dpi=FIG_DPI,
)
print(f"Saved {len(figure_paths)} figures")
display(pd.DataFrame({"figure": [str(p) for p in figure_paths]}).head(80))


Saved 43 movement-cycle hand-angle figures
Saved 15 figures


,figure
0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
2,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
3,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
4,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
5,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
6,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
7,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
8,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
9,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...


In [18]:
# -------------------------------------------------------------------
# 11b. Per-individual full reports (group runs)
# -------------------------------------------------------------------
#
# In a group run, each individual's results/<subject>/ folder receives only the
# per-subject *scoped* figures; every helper's AGGREGATE figures
# (velocity_vs_time_by_stiffness, average_velocity_vs_stiffness,
# all_velocity_vs_time_by_stiffness, *_acceleration_by_finger,
# time_magnitude / time_speed / time_radial_velocity /
# time_tangential_velocity, success_by_dominant_direction,
# success_vs_distance_from_center, the side_z_lift_* panels,
# the motor-control grids, and the success/Z + trajectory-distance
# contrasts) are produced once at the group level.
#
# This re-runs the ENTIRE per-selection figure pipeline for EVERY
# subject—restricted to that subject's rows—into results/<subject>/,
# so each individual folder matches a standalone single-subject run.
#
# It reuses the already-computed frames (no tracking recompute,
# no video re-decode) and is a no-op for a single-subject selection.
#
# Runs before organize_kinematic_results_tree() so the new files get
# sorted into figures/<category>/.

individual_report_paths = ka.save_individual_subject_reports(
    RESULTS_ROOT,
    trial_kinematic_summary=trial_kinematic_summary,
    trajectory_time_bins=trajectory_time_bins,
    kinematic_samples=kinematic_samples,
    side_z_trial_summary=side_z_trial_summary,
    side_z_samples=side_z_samples,
    side_z_group_time_summary=side_z_group_time_summary,
    n_time_bins=TRAJECTORY_TIME_BINS,
    fig_dpi=FIG_DPI,
)

print(
    f"Per-individual full reports: "
    f"{sum(len(v) for v in individual_report_paths.values())} figures across "
    f"{len(individual_report_paths)} subjects"
)


c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\A

Per-individual full reports: 4476 figures across 40 subjects


## 13. All-interactions kinematic folder

This cell writes only the `all` interaction-analysis branch for the normal kinematics results.

- `results/all` (`all`): all standard + comparison interaction segments.

The `standard` and `Comparison` split outputs are not created here; use dedicated analysis functions when those specific subsets are needed.


In [19]:
interaction_folder_index = ka.save_interaction_filtered_kinematic_outputs(
    OUTPUT_ROOT,
    trial_kinematic_summary=trial_kinematic_summary,
    trajectory_time_bins=trajectory_time_bins,
    pair_kinematic_summary=pair_kinematic_summary,
    side_z_trial_summary=side_z_trial_summary,
    side_z_samples=group_output_side_z_samples,
    fig_dpi=FIG_DPI,
    save_figures=True,
    save_sample_tables=False,  # keep large raw sample tables only in the legacy flat results folder
)
display(interaction_folder_index)


c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\user\A

,interaction_scope,interaction_code,folder_name,description,path,n_trial_segments,n_time_bins,n_subjects,manifest_exists
0,all,all,all,All stiffness interactions already used by the...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,20480,1021379,40,True


## 12. Manifest and interpretation notes

- `kinematic_samples.csv`: sample-level XY time series with raw pixel columns plus setup-scaled centimeter position, velocity, acceleration, jerk, and direction columns.
- `trial_kinematic_summary.csv`: per-trial features for modeling success, with cm metrics as the default physical analysis columns and px columns retained for traceability.
- `direction_success_summary.csv`: direction bins vs success.
- `distance_success_summary.csv`: distance-from-center vs success using setup-scaled centimeters.
- `kinematic_within_subject.csv`: subject-preserving table with centered/z-scored kinematic metrics.
- `within_finger_stiffness_effects.csv`: per-subject stiffness slopes by finger.
- Side-camera Z/lift outputs include raw pixel/proxy columns and setup-scaled cm proxy columns; without a side-camera calibration target, Z cm should be interpreted as a physical proxy rather than a true calibrated 3D reconstruction.


In [20]:
manifest = ka.analysis_manifest(OUTPUT_ROOT)
display(manifest)
print("Figures:", OUTPUT_ROOT / "figures")


,output,exists,path
0,trial_file_manifest.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
1,experiment_setup_context.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
2,kinematic_samples.parquet,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
3,pair_kinematic_summary.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
4,trial_kinematic_summary.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
...,...,...,...
83,side_z_trial_summary.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
84,side_z_group_time_summary.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
85,side_z_subject_stiffness_summary.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
86,side_z_by_stiffness_summary.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...


Figures: C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\result_fillter\L_N_E\figures


In [21]:
# --- 14. Per-subject csv tree (mirrors psychophysics save_selected_analysis_tree) ---
# Split every per-subject table (one carrying a subject_id column) by subject into
# results/<subject>/csv/. Tables without a subject_id are written to the selected
# subject/group owner folder and categorized by organize_kinematic_results_tree.
# Per-subject and aggregate figures were already written during the run by the
# figure helpers, so this only reorganizes the in-memory tables (no recompute).
per_subject_tables = {
    "trial_kinematic_summary.csv": trial_kinematic_summary,
    "pair_kinematic_summary.csv": pair_kinematic_summary,
    "trajectory_time_bins.csv": trajectory_time_bins,
    "direction_success_summary.csv": direction_success_summary,
    "distance_success_summary.csv": distance_success_summary,
    "subject_kinematic_summary.csv": subject_kinematic_summary,
    "participant_stiffness_kinematic_summary.csv": participant_stiffness_kinematic_summary,
    "kinematic_within_subject.csv": kinematic_within_subject,
    "within_finger_stiffness_effects.csv": within_finger_stiffness_effects,
    "finger_comparison_paired.csv": finger_comparison_paired,
    "trial_success_kinematic_z_table.csv": trial_success_kinematic_z_table,
    "success_kinematic_z_contrast_by_subject_finger.csv": success_kinematic_z_contrast_by_subject_finger,
    "subject_finger_trajectory.csv": subject_finger_trajectory,
    "subject_xy_trajectory.csv": subject_xy_trajectory,
    "subject_spatial_trajectory_summary.csv": subject_spatial_trajectory_summary,
    "subject_finger_spatial_distance.csv": subject_finger_spatial_distance,
    "subject_velocity_acceleration_profile.csv": subject_velocity_acceleration_profile,
    "subject_velocity_acceleration_summary.csv": subject_velocity_acceleration_summary,
    "subject_finger_velocity_acceleration_distance.csv": subject_finger_velocity_acceleration_distance,
    "trial_3d_kinematic_summary.csv": trial_3d_kinematic_summary,
    "subject_3d_kinematic_summary.csv": subject_3d_kinematic_summary,
    "hand_orientation_plane_summary.csv": hand_orientation_plane_summary,
    "side_z_trial_summary.csv": side_z_trial_summary,
    "side_z_subject_stiffness_summary.csv": side_z_subject_stiffness_summary,
}
per_subject_tree_manifest = ka.save_selected_kinematic_tree(RUN_OUTPUT_ROOT, tables=per_subject_tables)
_n_subject = int((per_subject_tree_manifest["kind"] == "subject").sum())
_n_aggregate = int((per_subject_tree_manifest["kind"] == "aggregate").sum())
print(f"Per-subject csv files written: {_n_subject}; aggregate csv files: {_n_aggregate}")
print("Per-subject tree base:", RESULTS_ROOT.resolve())
display(per_subject_tree_manifest.head(60))


C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\kinematics_analysis.py:784: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, encoding=enc, **kwargs)


Per-subject csv files written: 3400; aggregate csv files: 2
Per-subject tree base: C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\result_fillter


,table,kind,path
0,trial_kinematic_summary.csv,subject,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
1,trial_kinematic_summary.csv,subject,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
2,trial_kinematic_summary.csv,subject,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
3,trial_kinematic_summary.csv,subject,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
4,trial_kinematic_summary.csv,subject,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
5,trial_kinematic_summary.csv,subject,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
6,trial_kinematic_summary.csv,subject,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
7,trial_kinematic_summary.csv,subject,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
8,trial_kinematic_summary.csv,subject,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
9,trial_kinematic_summary.csv,subject,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...


In [22]:
# --- 15. Organize the results tree by subject + category ---
# Reorganize everything this run produced into the final layout:
#   results/<owner>/figures/<category>/...   and   results/<owner>/csv/<category>/...
# where <owner> is each subject plus the requested aggregate owner. Named
# group/combined runs use L_E / N_E / L_N_E top-level siblings; hand-picked
# subject lists use one custom aggregate folder and are not widened to full groups.
# Single-subject runs stay as one
# folder. Categories: s_vs_c_pos, s_vs_c_v, s_vs_c_a, hand_orientation,
# movement_orientation, z_lift, trajectories, velocity, acceleration, success, other.
organize_log = ka.organize_kinematic_results_tree(RESULTS_ROOT, RUN_OUTPUT_ROOT, DATA_SELECTION)
print(f"Reorganized {len(organize_log)} files into the by-subject / by-category tree.")
print("Owners:", sorted(organize_log["owner"].unique()) if not organize_log.empty else "(none)")
display(organize_log.groupby("owner").size().reset_index(name="files"))


Reorganized 8324 files into the by-subject / by-category tree.
Owners: ['L_E', 'L_E_1', 'L_E_10', 'L_E_11', 'L_E_12', 'L_E_14', 'L_E_15', 'L_E_16', 'L_E_17', 'L_E_18', 'L_E_2', 'L_E_20', 'L_E_21', 'L_E_22', 'L_E_3', 'L_E_4', 'L_E_5', 'L_E_6', 'L_E_7', 'L_E_8', 'L_E_9', 'L_N_E', 'N_E', 'N_E_1', 'N_E_11', 'N_E_12', 'N_E_13', 'N_E_14', 'N_E_15', 'N_E_16', 'N_E_17', 'N_E_18', 'N_E_19', 'N_E_2', 'N_E_20', 'N_E_21', 'N_E_3', 'N_E_4', 'N_E_5', 'N_E_6', 'N_E_7', 'N_E_8', 'N_E_9']


,owner,files
0,L_E,45
1,L_E_1,151
2,L_E_10,151
3,L_E_11,151
4,L_E_12,151
5,L_E_14,151
6,L_E_15,151
7,L_E_16,151
8,L_E_17,151
9,L_E_18,151


## 16. Filtered output sanity summary

Run this after the organization cell to confirm that `result_fillter` contains filtered z-tracking inputs and categorized outputs.


In [23]:
def list_existing_category_dirs(root: Path) -> pd.DataFrame:
    rows = []
    for owner in sorted([p for p in root.iterdir() if p.is_dir()]):
        if owner.name == "filtered_z_tracking_sources":
            continue
        for kind in ["csv", "figures"]:
            base = owner / kind
            if not base.exists():
                continue
            for category in sorted([p for p in base.iterdir() if p.is_dir()]):
                rows.append({
                    "owner": owner.name,
                    "kind": kind,
                    "category": category.name,
                    "files": sum(1 for p in category.rglob("*") if p.is_file()),
                    "filter_tag": FILTER_TAG,
                })
    return pd.DataFrame(rows)


category_manifest = list_existing_category_dirs(RESULTS_ROOT)
ka.save_csv(category_manifest, OUTPUT_ROOT, "category_manifest_fillter.csv")
print({
    "results_root": str(RESULTS_ROOT),
    "run_output_root": str(OUTPUT_ROOT),
    "filtered_z_tracking_root": str(FILTERED_Z_TRACKING_ROOT),
    "filtered_z_tracking_files": len(list(FILTERED_Z_TRACKING_ROOT.rglob("z_tracking_fillter.csv"))),
    "category_rows": len(category_manifest),
    "filter_tag": FILTER_TAG,
})
display(category_manifest.head(100))


{'results_root': 'C:\\Users\\user\\BIO MEDICAL ROBOTICS Dropbox\\Elisheva Shiri Decktor\\BGU\\Codes\\Parallel_Heptics\\analysis\\Kinematics\\result_fillter', 'run_output_root': 'C:\\Users\\user\\BIO MEDICAL ROBOTICS Dropbox\\Elisheva Shiri Decktor\\BGU\\Codes\\Parallel_Heptics\\analysis\\Kinematics\\result_fillter\\L_N_E', 'filtered_z_tracking_root': 'C:\\Users\\user\\BIO MEDICAL ROBOTICS Dropbox\\Elisheva Shiri Decktor\\BGU\\Codes\\Parallel_Heptics\\analysis\\Kinematics\\result_fillter\\filtered_z_tracking_sources\\L_N_E_fillter', 'filtered_z_tracking_files': 10240, 'category_rows': 469, 'filter_tag': 'fillter'}


,owner,kind,category,files,filter_tag
0,L_E,csv,other,8,fillter
1,L_E,csv,success,3,fillter
2,L_E,csv,trajectories,3,fillter
3,L_E,csv,velocity,2,fillter
4,L_E,csv,z_lift,2,fillter
...,...,...,...,...,...
95,L_E_17,figures,trajectories,23,fillter
96,L_E_17,figures,velocity,28,fillter
97,L_E_18,csv,hand_orientation,1,fillter
98,L_E_18,csv,other,9,fillter
